# HM-RAG + MAMMQA: frozen VALIDATION16_V2 experiment

This notebook runs the already-repaired pipeline on the deterministic, disjoint 16-question
`VALIDATION16_V2` set. The QA implementation and inference settings are frozen: this is a
measurement run, not a tuning run.

Predictions are checkpointed after every question. Gold answers and support annotations are
materialized only after all 16 predictions have been frozen.

In [ ]:
# Cell 1 - environment diagnostics; allocator configured before importing torch
import importlib.util, os, platform, subprocess, sys
os.environ.setdefault("PYTORCH_ALLOC_CONF","expandable_segments:True")
print("Python:",sys.version)
try:
    import torch
    print("PyTorch:",torch.__version__,"CUDA:",torch.cuda.is_available(),"GPUs:",torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):print(i,torch.cuda.get_device_name(i))
except Exception as e:print("PyTorch diagnostic failed:",e)

In [ ]:
# Cell 2 - install only missing, released packages
REQUIRED = {
    "transformers": "transformers>=4.49,<5", "accelerate": "accelerate>=1,<2",
    "bitsandbytes": "bitsandbytes>=0.45,<1", "qwen_vl_utils": "qwen-vl-utils>=0.0.8",
    "sentence_transformers": "sentence-transformers>=3,<6", "faiss": "faiss-cpu>=1.8,<2",
    "networkx": "networkx>=3,<4", "pandas": "pandas>=2,<3", "tabulate": "tabulate>=0.9,<1",
    "rank_bm25": "rank-bm25>=0.2.2,<1",
}
missing = [spec for module, spec in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else: print("All optional dependencies are already installed.")

In [ ]:
# Cell 3 - reproducible runtime configuration and isolated output/cache paths
import ast, copy, csv, gc, hashlib, json, math, os, pickle, random, re, string, time, zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Sequence, Tuple
import faiss, networkx as nx, numpy as np, pandas as pd, torch
from PIL import Image
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

NOTEBOOK_START_TIME=time.perf_counter()
SEED=42; RUN_MODE="smoke"; RUN_RETRIEVAL_ONLY=False
EXPERIMENT_PHASE="VALIDATION16_V2"  # frozen measurement run; do not tune mid-run
FAST_VALIDATION=EXPERIMENT_PHASE=="ENGINEERING_GATE"
NUM_SHARDS=1; SHARD_ID=0; VALIDATION_SELECTION_SEED=20260924
EXPERIMENT_ID={"ENGINEERING_GATE":"qa_targeted_repair_v2_gate_seed42","DEBUG16_REGRESSION":"qa_targeted_repair_v2_debug16_seed42","VALIDATION16_V2":"qa_targeted_repair_v2_validation16_seed42"}[EXPERIMENT_PHASE]
FAILED_EXPERIMENT_ID="qa_smoke16_full_pipeline_seed42"
MODEL_NAME="Qwen/Qwen2.5-VL-7B-Instruct"
EMBED_MODEL="BAAI/bge-small-en-v1.5"; CLIP_MODEL="openai/clip-vit-base-patch32"
RETRIEVAL_MODE="hybrid"; ROUTER_MODE="soft"
RETRIEVAL_CANDIDATE_K=20; MIN_METRIC_CANDIDATE_K=10; FALLBACK_RETRIEVAL_K=20
TEXT_CONTEXT_K=3; TABLE_CONTEXT_K=2; IMAGE_CONTEXT_K=2
TEXT_CONTEXT_K_MAX=4; TABLE_CONTEXT_K_MAX=3; IMAGE_CONTEXT_K_MAX=2
FALLBACK_TEXT_K=1; FALLBACK_TABLE_K=1; FALLBACK_IMAGE_K=1
FEEDBACK_CANDIDATE_K=20; CONTEXT_DUPLICATE_THRESHOLD=0.72
CONTEXT_SELECTION_WEIGHTS={"relevance":1.0,"original_question":0.35,"uncovered_query":0.25,"modality_need":0.15,"graph":0.10,"feedback":0.30,"redundancy":0.35}
ROUTER_CONFIDENCE_THRESHOLD=0.65; PLANNER_MODALITY_THRESHOLD=0.55
RRF_K=60; GRAPH_SEED_K=10; GRAPH_MAX_EXPANDED_DOCS=50; GRAPH_MAX_DEGREE=50
TEXT_TOP_K=TEXT_CONTEXT_K; TABLE_TOP_K=TABLE_CONTEXT_K; IMAGE_TOP_K=IMAGE_CONTEXT_K; IMAGE_CANDIDATES=50
MAX_RETRIEVAL_ROUNDS=2; ENABLE_GRAPH=True; ENABLE_WEB=False
IMAGE_RETRIEVAL_MODE="hybrid"; AGGREGATOR_SEES_QUESTION=True
TABLE_CHUNK_ROWS=50; LARGE_TABLE_ROWS=120; IMAGE_REASONING_USES_TITLE=False

MAX_INPUT_TOKENS=4096
STAGE_MAX_NEW_TOKENS={"planner":64,"stage1":96,"stage2":128,"verifier":64,"aggregator":48}
FORMAT_REPAIR_NEW_TOKENS=48; REPAIR_STAGES={"aggregator"}
STAGE1_EVIDENCE_CHARS=1600; STAGE2_EVIDENCE_CHARS=1200; VERIFIER_EVIDENCE_CHARS=900
MAMMQA_TEMPERATURE=0.3; MAMMQA_TOP_P=0.7; MAMMQA_DO_SAMPLE=True
PLANNER_DO_SAMPLE=False; VERIFIER_DO_SAMPLE=False; IMAGE_PROCESSOR_USE_FAST=False
CLIP_GPU_MIN_FREE_GIB=6.0; BGE_GPU_MIN_FREE_GIB=3.0; CLIP_DEVICE="cpu"; BGE_DEVICE="cpu"

BASE_OUTPUT_ROOT=Path("/kaggle/working/hm_mammqa")
OUTPUT_ROOT=BASE_OUTPUT_ROOT/EXPERIMENT_ID
FAILED_OUTPUT_ROOT=BASE_OUTPUT_ROOT/FAILED_EXPERIMENT_ID
CACHE_EXPORT_ROOT=OUTPUT_ROOT/"persistent_cache_package"
DIRS={"cache":CACHE_EXPORT_ROOT/"cache","indexes":CACHE_EXPORT_ROOT/"indexes","images":BASE_OUTPUT_ROOT/"images",
      "predictions":OUTPUT_ROOT/"predictions","metrics":OUTPUT_ROOT/"metrics","logs":OUTPUT_ROOT/"logs",
      "diagnostics":OUTPUT_ROOT/"diagnostics","offload":OUTPUT_ROOT/"offload"}
DIRS["retrieval_logs"]=DIRS["diagnostics"]/"retrieval_logs"
for p in set(DIRS.values()):p.mkdir(parents=True,exist_ok=True)

input_root=Path("/kaggle/input"); cache_roots=[]
if input_root.exists():
    # Accept direct, nested, renamed, or previously published Kaggle cache datasets.
    for marker in list(input_root.rglob("text_faiss.index"))+list(input_root.rglob("image_manifest.json")):
        parent=marker.parent; root=parent.parent if parent.name in {"indexes","cache"} else parent
        cache_roots.append(root)
    for p in input_root.iterdir():
        cache_roots.extend([p,p/"persistent_cache_package",p/"hm_mammqa"])
cache_roots += [BASE_OUTPUT_ROOT]
CACHE_READ_ROOTS=list(dict.fromkeys(p.resolve() for p in cache_roots if p.exists() and ((p/"indexes").exists() or (p/"cache").exists())))
print("Persistent cache roots discovered:",[str(p) for p in CACHE_READ_ROOTS])
CACHE_STATUS={}  # reporting only; never changes retrieval semantics

def artifact_group(specs):
    for root in CACHE_READ_ROOTS:
        paths={key:root/kind/name for key,kind,name in specs}
        if all(p.exists() for p in paths.values()):return paths,True,root
    paths={key:CACHE_EXPORT_ROOT/kind/name for key,kind,name in specs}
    for p in paths.values():p.parent.mkdir(parents=True,exist_ok=True)
    return paths,False,CACHE_EXPORT_ROOT

def artifact_output_group(specs):
    paths={key:CACHE_EXPORT_ROOT/kind/name for key,kind,name in specs}
    for p in paths.values():p.parent.mkdir(parents=True,exist_ok=True)
    return paths

GENERATION_DIAGNOSTICS_PATH=DIRS["logs"]/"generation_diagnostics.jsonl"
PLANNER_DIAGNOSTICS_PATH=DIRS["logs"]/"planner_diagnostics.jsonl"
TRUNCATION_DIAGNOSTICS_PATH=DIRS["logs"]/"truncation_diagnostics.jsonl"
TIMING_LOG_PATH=DIRS["logs"]/"timing.jsonl"
RUNTIME_CONFIG_PATH=DIRS["logs"]/"runtime_config.json"
OPTIMIZATION_SUMMARY_PATH=DIRS["logs"]/"runtime_optimization_summary.json"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
assert SEED==42 and RUN_MODE in {"smoke","full"} and RUN_RETRIEVAL_ONLY is False
assert RETRIEVAL_CANDIDATE_K>=MIN_METRIC_CANDIDATE_K>=10
assert MAX_RETRIEVAL_ROUNDS==2 and not ENABLE_WEB and IMAGE_REASONING_USES_TITLE is False
assert NUM_SHARDS>=1 and 0<=SHARD_ID<NUM_SHARDS and REPAIR_STAGES=={"aggregator"}
assert EXPERIMENT_PHASE in {"ENGINEERING_GATE","DEBUG16_REGRESSION","VALIDATION16_V2"}

def gpu_memory(label):
    if not torch.cuda.is_available():print(label,"CPU only"); return []
    stats=[]; raw=[]
    for i in range(torch.cuda.device_count()):
        free,total=torch.cuda.mem_get_info(i); raw.append((free,total)); stats.append(f"GPU{i} free={free/2**30:.2f}/{total/2**30:.2f}GiB")
    print(label," | ".join(stats)); return raw
gpu_memory("startup:")

In [ ]:
# Cell 4 - confirmed MMQA paths with unambiguous fallback
DATA_ROOT=Path("/kaggle/input/datasets/adibraian/multimodalqa-dataset")
NAMES={
 "train":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_train.jsonl",
 "dev":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_dev.jsonl",
 "test":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_test.jsonl",
 "texts":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_texts.jsonl",
 "tables":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_tables.jsonl",
 "images":"multimodalqa_final_dataset_pipeline_camera_ready_MMQA_images.jsonl"}
FOLDERS={k:f"MMQA_{k}.jsonl" for k in NAMES}
def resolve_dataset_file(key:str)->Path:
    primary=DATA_ROOT/FOLDERS[key]/NAMES[key]
    if primary.is_file(): return primary
    roots=[DATA_ROOT,Path("/kaggle/input")]
    matches=sorted({p.resolve() for root in roots if root.exists() for p in root.rglob(NAMES[key])})
    if len(matches)!=1: raise FileNotFoundError(f"{key}: expected one {NAMES[key]}, found {matches}")
    return matches[0]
PATHS={k:resolve_dataset_file(k) for k in NAMES}
for k,p in PATHS.items(): print(f"{k:7s} {p}")

In [ ]:
# Cell 5 - JSONL loading
def load_jsonl(path:Path)->List[dict]:
    out=[]
    with path.open(encoding="utf-8") as f:
        for no,line in enumerate(f,1):
            if line.strip():
                try: out.append(json.loads(line))
                except json.JSONDecodeError as e: raise ValueError(f"{path}:{no}: {e}") from e
    return out
dev_raw=load_jsonl(PATHS["dev"]); texts_raw=load_jsonl(PATHS["texts"])
tables_raw=load_jsonl(PATHS["tables"]); images_raw=load_jsonl(PATHS["images"])
print(len(dev_raw),len(texts_raw),len(tables_raw),len(images_raw))

In [ ]:
# Cell 6 - schema inspection and hard inference/evaluation separation
for name,row in [("question",dev_raw[0]),("text",texts_raw[0]),("table",tables_raw[0]),("image",images_raw[0])]:
    print(name,{k:type(v).__name__ for k,v in row.items()})
@dataclass(frozen=True)
class QueryInput: qid:str; question:str
@dataclass(frozen=True)
class EvaluationOnly:
    gold_answers:List[str]; gold_support_ids:List[str]; gold_support_modalities:Dict[str,str]
    metadata_type:str="unknown"; gold_modalities:List[str]=field(default_factory=list)
    intermediate_answers:Any=None
FORBIDDEN={"answers","supporting_context","metadata","table_id","text_doc_ids","image_doc_ids",
           "modalities","type","intermediate_answers","gold_answers","gold_support_ids"}
def assert_safe_query(q:QueryInput):
    assert isinstance(q,QueryInput) and set(asdict(q))=={"qid","question"}
    assert not (set(asdict(q)) & FORBIDDEN)
def query_only(row:dict)->QueryInput:
    q=QueryInput(str(row.get("qid",row.get("id",""))),str(row["question"]))
    assert_safe_query(q); return q
def assert_inference_boundary(fn_name:str,q:QueryInput):
    assert_safe_query(q)
    assert set(asdict(q))=={"qid","question"},f"{fn_name} received a non-query field"
print("LEAKAGE CHECK PASSED")

In [ ]:
# Cell 7 - cached official-image manifest, ZIP fallback and pixel validation
def safe_extract(zp:Path,dst:Path):
    with zipfile.ZipFile(zp) as z:
        root=dst.resolve()
        for member in z.infolist():
            target=(dst/member.filename).resolve()
            if root not in target.parents and target!=root:raise ValueError(f"Unsafe ZIP member: {member.filename}")
        z.extractall(dst)

def load_image_manifest(records):
    paths,found,root=artifact_group([("manifest","cache","image_manifest.json")])
    manifest_path=paths["manifest"]
    if found:
        try:
            payload=json.loads(manifest_path.read_text(encoding="utf-8")); mapping={}
            for image_id,rel in payload.get("paths",{}).items():
                p=input_root/rel
                if p.is_file():mapping[str(image_id)]=p
            if len(mapping)==len(records):
                CACHE_STATUS["image_manifest"]="HIT"; print("CACHE HIT / BUILT image_manifest: HIT",manifest_path); return mapping
        except Exception as e:print("Image manifest rejected; rebuilding:",e)
    return None

def discover_image_paths(records):
    cached=load_image_manifest(records)
    if cached is not None:return cached
    expected={Path(str(r.get("path",""))).name for r in records if r.get("path")}
    mounts=input_root; extract=DIRS["images"]/"official"
    named=[] if not mounts.exists() else [p for p in mounts.rglob("final_dataset_images") if p.is_dir()]
    zips=[] if not mounts.exists() else [p for p in mounts.rglob("*.zip") if re.search(r"(mmqa|multimodal).*image|image.*(mmqa|multimodal)",p.name,re.I)]
    valid_zips=[]
    for z in zips:
        try:
            with zipfile.ZipFile(z) as zh:
                if any(Path(n).name in expected for n in zh.namelist()):valid_zips.append(z)
        except zipfile.BadZipFile:pass
    if not named and valid_zips:
        if len(valid_zips)>1:raise RuntimeError(f"Ambiguous official image ZIPs: {valid_zips}")
        marker=extract/".complete"
        if not marker.exists():extract.mkdir(parents=True,exist_ok=True); safe_extract(valid_zips[0],extract); marker.touch()
        named=[extract]
    roots=named or ([mounts] if mounts.exists() else []); by_name={}
    for root in roots:
        for p in root.rglob("*"):
            if p.is_file() and p.name in expected:
                if p.name in by_name and by_name[p.name]!=p:raise RuntimeError(f"Ambiguous image filename {p.name}")
                by_name[p.name]=p
    mapping={str(r["id"]):by_name[Path(str(r.get("path",""))).name] for r in records if Path(str(r.get("path",""))).name in by_name}
    if len(mapping)==len(records) and all(input_root==p or input_root in p.parents for p in mapping.values()):
        out=CACHE_EXPORT_ROOT/"cache"/"image_manifest.json"; out.parent.mkdir(parents=True,exist_ok=True)
        out.write_text(json.dumps({"count":len(mapping),"paths":{k:str(v.relative_to(input_root)) for k,v in mapping.items()}},indent=2),encoding="utf-8")
        CACHE_STATUS["image_manifest"]="BUILT"; print("CACHE HIT / BUILT image_manifest: BUILT",out)
    return mapping

IMAGE_PATHS=discover_image_paths(images_raw)
if "image_manifest" not in CACHE_STATUS:
    CACHE_STATUS["image_manifest"]="BUILT_IN_MEMORY"; print("CACHE HIT / BUILT image_manifest: BUILT_IN_MEMORY")
assert len(images_raw)==57058,f"Expected 57,058 official MMQA images, found metadata for {len(images_raw)}"
assert len(IMAGE_PATHS)==57058,f"Expected 57,058 resolved official images, found {len(IMAGE_PATHS)}"
for p in list(IMAGE_PATHS.values())[:5]:
    with Image.open(p) as im:im.verify()
print(f"official image manifest verified: {len(IMAGE_PATHS):,}/57,058; sample PIL checks=5")
FULL_IMAGE_BENCHMARK=True

In [ ]:
# Cell 8 - corpus normalization and table representations
def cell_text(c:Any)->str:
    if isinstance(c,dict): return str(c.get("text",c.get("value","")))
    return str(c)
def table_parts(row:dict)->Tuple[List[str],List[List[str]]]:
    t=row.get("table",row); header=t.get("header",[]); rows=t.get("table_rows",t.get("rows",[]))
    heads=[str(x.get("column_name",x.get("text",""))) if isinstance(x,dict) else str(x) for x in header]
    return heads,[[cell_text(c) for c in rr] for rr in rows]
def markdown_table(heads,rows)->str:
    width=max([len(heads),*(len(r) for r in rows)] or [0]); heads=(heads+[f"col_{i}" for i in range(len(heads),width)])[:width]
    clean=lambda s:str(s).replace("|","\\|").replace("\n"," ")
    return "\n".join(["| "+" | ".join(map(clean,heads))+" |","| "+" | ".join(["---"]*width)+" |",*["| "+" | ".join(map(clean,r+[""]*(width-len(r))))+" |" for r in rows]])
def chunk_table_rows(headers:List[str],rows:List[List[str]],max_rows=TABLE_CHUNK_ROWS,max_chars=6000):
    chunks=[]; start=0
    while start<len(rows):
        end=start; size=len(" | ".join(headers))
        while end<len(rows) and end-start<max_rows:
            row_size=sum(len(str(v)) for v in rows[end])+len(headers)*4
            if end>start and size+row_size>max_chars: break
            size+=row_size; end+=1
        chunks.append((start,max(start+1,end)))
        start=max(start+1,end)
    return chunks or [(0,0)]
def normalize_corpus():
    docs=[]
    for r in texts_raw:
        did=str(r["id"]); title=str(r.get("title","")); body=str(r.get("text",r.get("paragraph","")))
        docs.append({"doc_id":did,"chunk_id":did,"parent_doc_id":did,"parent_table_id":None,"modality":"text","title":title,"retrieval_text":f"{title}\n{body}","reasoning_content":body,"raw_content":body,"image_path":None,"row_start":None,"row_end":None,"headers":[]})
    for r in tables_raw:
        did=str(r["id"]); title=str(r.get("title",r.get("name",""))); h,rows=table_parts(r)
        spans=chunk_table_rows(h,rows)
        for start,end in spans:
            ch=rows[start:end]; cid=did if len(spans)==1 else f"{did}::rows_{start}-{max(start,end-1)}"
            linear=" ; ".join([f"{k}: {v}" for rr in ch for k,v in zip(h,rr)])
            neighbor_start=max(0,start-1); neighbor_end=min(len(rows),end+1)
            reasoning=markdown_table(h,rows[neighbor_start:neighbor_end])
            docs.append({"doc_id":cid,"chunk_id":cid,"parent_doc_id":did,"parent_table_id":did,"modality":"table","title":title,"retrieval_text":f"{title}\ncolumns: {' | '.join(h)}\n{linear}","reasoning_content":reasoning,"raw_content":reasoning,"image_path":None,"source_record":r,"row_start":start,"row_end":max(start,end-1),"headers":h})
    for r in images_raw:
        did=str(r["id"]); title=str(r.get("title",""))
        docs.append({"doc_id":did,"chunk_id":did,"parent_doc_id":did,"parent_table_id":None,"modality":"image","title":title,"retrieval_text":title,"reasoning_content":{"path":r.get("path")},"raw_content":{"path":r.get("path"),"url":r.get("url")},"image_path":str(IMAGE_PATHS[did]) if did in IMAGE_PATHS else None,"row_start":None,"row_end":None,"headers":[]})
    return docs
CORPUS=normalize_corpus(); BY_MODALITY={m:[d for d in CORPUS if d["modality"]==m] for m in ["text","table","image"]}
DOC_LOOKUP={(d["modality"],d["doc_id"]):d for d in CORPUS}
PARENT_LOOKUP=defaultdict(list)
for d in CORPUS:PARENT_LOOKUP[(d["modality"],d["parent_doc_id"])].append(d)
TABLE_PARENT_TO_CHUNKS={pid:chunks for (mod,pid),chunks in PARENT_LOOKUP.items() if mod=="table"}
print({m:len(v) for m,v in BY_MODALITY.items()},"table parents",len(TABLE_PARENT_TO_CHUNKS))
assert sum(len(v) for v in TABLE_PARENT_TO_CHUNKS.values())==len(BY_MODALITY["table"])
if not RUN_RETRIEVAL_ONLY:
    assert len(texts_raw)>200000 and len(tables_raw)>10000 and len(images_raw)>50000,"The full MMQA corpus was not loaded"
    assert len(BY_MODALITY["text"])==len(texts_raw) and len(TABLE_PARENT_TO_CHUNKS)==len(tables_raw) and len(BY_MODALITY["image"])==len(images_raw)
    print(f"FULL CORPUS CHECK PASSED: {len(texts_raw):,} text documents, {len(tables_raw):,} original tables, {len(images_raw):,} official images")

In [ ]:
# Cells 9-10 - persistent normalized FAISS/embedding/BM25 indexes
def fingerprint(records):
    h=hashlib.sha256(); h.update(EMBED_MODEL.encode())
    for d in records:h.update((d["doc_id"]+"\0"+d["retrieval_text"]).encode("utf-8"))
    return h.hexdigest()

class DenseIndex:
    def __init__(self,modality,records,encoder):self.modality,self.records,self.encoder=modality,records,encoder; self.index=None
    def build_or_load(self):
        specs=[("index","indexes",f"{self.modality}_faiss.index"),("mapping","indexes",f"{self.modality}_doc_mapping.json"),("embeddings","cache",f"{self.modality}_embeddings.npy")]
        paths,found,root=artifact_group(specs); fp=fingerprint(self.records)
        if found and json.loads(paths["mapping"].read_text())["fingerprint"]==fp:
            self.index=faiss.read_index(str(paths["index"])); CACHE_STATUS[f"{self.modality}_index"]="HIT"; print(f"CACHE HIT / BUILT {self.modality} index: HIT",root); return
        if found:paths=artifact_output_group(specs)
        emb=self.encoder.encode([d["retrieval_text"] for d in self.records],batch_size=128,normalize_embeddings=True,show_progress_bar=True).astype("float32")
        np.save(paths["embeddings"],emb); self.index=faiss.IndexFlatIP(emb.shape[1]); self.index.add(emb); faiss.write_index(self.index,str(paths["index"]))
        paths["mapping"].write_text(json.dumps({"fingerprint":fp,"ids":[d["doc_id"] for d in self.records]}))
        CACHE_STATUS[f"{self.modality}_index"]="BUILT"; print(f"CACHE HIT / BUILT {self.modality} index: BUILT",CACHE_EXPORT_ROOT)
    def search(self,query,k,subquery="",query_source="original_question"):
        q=self.encoder.encode([query],normalize_embeddings=True).astype("float32"); scores,idx=self.index.search(q,min(k,len(self.records)))
        return [{**self.records[i],"dense_rank":rank,"dense_score":float(s),"bm25_rank":None,"bm25_score":None,"hybrid_rrf_score":0.0,"graph_rank":None,"graph_score":0.0,"final_score":float(s),"retrieved_by_subquery":[subquery or query],"query_provenance":[{"query_text":subquery or query,"query_source":query_source,"rank":rank,"score":float(s),"method":"dense"}]} for rank,(s,i) in enumerate(zip(scores[0],idx[0]),1) if i>=0]

def bm25_tokens(text):return re.findall(r"[a-z0-9]+",str(text).lower())
class BM25Index:
    def __init__(self,modality,records):self.modality,self.records=modality,records; self.index=None
    def build_or_load(self):
        specs=[("bm25","indexes",f"{self.modality}_bm25.pkl")]; paths,found,root=artifact_group(specs); fp=fingerprint(self.records)
        if found:
            payload=pickle.loads(paths["bm25"].read_bytes())
            if payload.get("fingerprint")==fp:self.index=payload["index"]; CACHE_STATUS[f"{self.modality}_bm25"]="HIT"; print(f"CACHE HIT / BUILT {self.modality} BM25: HIT",root); return
            paths=artifact_output_group(specs)
        self.index=BM25Okapi([bm25_tokens(d["retrieval_text"]) for d in self.records]); paths["bm25"].write_bytes(pickle.dumps({"fingerprint":fp,"index":self.index},pickle.HIGHEST_PROTOCOL))
        CACHE_STATUS[f"{self.modality}_bm25"]="BUILT"; print(f"CACHE HIT / BUILT {self.modality} BM25: BUILT",CACHE_EXPORT_ROOT)
    def search(self,query,k,subquery="",query_source="original_question"):
        scores=np.asarray(self.index.get_scores(bm25_tokens(query))); k=min(k,len(scores)); idx=np.argpartition(scores,-k)[-k:] if 0<k<len(scores) else np.arange(len(scores)); idx=idx[np.argsort(scores[idx])[::-1]][:k]
        return [{**self.records[i],"dense_rank":None,"dense_score":None,"bm25_rank":rank,"bm25_score":float(scores[i]),"hybrid_rrf_score":0.0,"graph_rank":None,"graph_score":0.0,"final_score":float(scores[i]),"retrieved_by_subquery":[subquery or query],"query_provenance":[{"query_text":subquery or query,"query_source":query_source,"rank":rank,"score":float(scores[i]),"method":"bm25"}]} for rank,i in enumerate(idx,1)]

gpu_memory("before BGE:")
QUERY_ENCODER=SentenceTransformer(EMBED_MODEL,device="cuda:0" if torch.cuda.is_available() else "cpu")
TEXT_INDEX=DenseIndex("text",BY_MODALITY["text"],QUERY_ENCODER); TEXT_INDEX.build_or_load()
TABLE_INDEX=DenseIndex("table",BY_MODALITY["table"],QUERY_ENCODER); TABLE_INDEX.build_or_load()
IMAGE_TITLE_INDEX=DenseIndex("image_titles",BY_MODALITY["image"],QUERY_ENCODER); IMAGE_TITLE_INDEX.build_or_load()
QUERY_ENCODER.to("cpu"); BGE_DEVICE="cpu"
for index in [TEXT_INDEX,TABLE_INDEX,IMAGE_TITLE_INDEX]:index.encoder=QUERY_ENCODER
BM25_INDEXES={m:BM25Index(m,BY_MODALITY[m]) for m in ["text","table","image"]}
for index in BM25_INDEXES.values():index.build_or_load()

def rrf_fuse(dense,lexical,k):
    merged={}
    for source,items in [("dense",dense),("bm25",lexical)]:
        for rank,d in enumerate(items,1):
            key=d["doc_id"]; x=merged.setdefault(key,{**d,"dense_rank":None,"dense_score":None,"bm25_rank":None,"bm25_score":None,"retrieved_by_subquery":[],"query_provenance":[]})
            x[f"{source}_rank"]=rank; x[f"{source}_score"]=d.get(f"{source}_score")
            x["retrieved_by_subquery"]=list(dict.fromkeys(x["retrieved_by_subquery"]+d.get("retrieved_by_subquery",[])))
            x["query_provenance"] += [p for p in d.get("query_provenance",[]) if p not in x["query_provenance"]]
    for x in merged.values():
        x["hybrid_rrf_score"]=(1/(RRF_K+x["dense_rank"]) if x["dense_rank"] else 0)+(1/(RRF_K+x["bm25_rank"]) if x["bm25_rank"] else 0); x["final_score"]=x["hybrid_rrf_score"]
    return sorted(merged.values(),key=lambda x:x["hybrid_rrf_score"],reverse=True)[:k]

def search_modality(modality,query,k,mode,subquery,query_source="original_question"):
    dense_index={"text":TEXT_INDEX,"table":TABLE_INDEX,"image":IMAGE_TITLE_INDEX}[modality]; internal=max(k*3,50)
    if mode=="dense":return dense_index.search(query,k,subquery,query_source)
    if mode=="bm25":return BM25_INDEXES[modality].search(query,k,subquery,query_source)
    return rrf_fuse(dense_index.search(query,internal,subquery,query_source),BM25_INDEXES[modality].search(query,internal,subquery,query_source),k)

In [ ]:
# Cell 11 - official-image CLIP reranking with persistent cache and safe GPU fallback
class ClipReranker:
    def __init__(self):self.model=None; self.processor=None; self.cache={}; self.actual_device="cpu"
    @staticmethod
    def unwrap_features(output):
        if isinstance(output,torch.Tensor):return output
        pooled=getattr(output,"pooler_output",None)
        if isinstance(pooled,torch.Tensor):return pooled
        if isinstance(output,(tuple,list)):
            tensors=[x for x in output if isinstance(x,torch.Tensor) and x.ndim==2]
            if tensors:return tensors[-1]
        raise TypeError(f"Unsupported CLIP feature output: {type(output).__name__}")
    def load(self):
        from transformers import CLIPModel,CLIPProcessor
        self.processor=CLIPProcessor.from_pretrained(CLIP_MODEL,use_fast=False)
        target=CLIP_DEVICE
        try:self.model=CLIPModel.from_pretrained(CLIP_MODEL).eval().to(target); self.actual_device=target
        except torch.cuda.OutOfMemoryError:
            gc.collect(); torch.cuda.empty_cache(); self.model=CLIPModel.from_pretrained(CLIP_MODEL).eval().to("cpu"); self.actual_device="cpu"
        specs=[("clip","cache","clip_embeddings.pkl")]; paths,found,root=artifact_group(specs)
        if found:self.cache=pickle.loads(paths["clip"].read_bytes()); print("Loaded persistent CLIP cache from",root)
        self.cache_path=artifact_output_group(specs)["clip"]
        print("CLIP actual device =",self.actual_device)
    def encode_missing(self,docs):
        if self.model is None:self.load()
        missing=[d for d in docs if d.get("image_path") and d["doc_id"] not in self.cache]
        for start in range(0,len(missing),32):
            batch=missing[start:start+32]; ims=[]
            for d in batch:
                with Image.open(d["image_path"]) as im:ims.append(im.convert("RGB"))
            x=self.processor(images=ims,return_tensors="pt").to(self.model.device)
            with torch.inference_mode():z=self.unwrap_features(self.model.get_image_features(**x)); z=z/z.norm(dim=-1,keepdim=True)
            self.cache.update({d["doc_id"]:e.cpu().numpy().astype("float32") for d,e in zip(batch,z)}); del x,z
        if missing:self.cache_path.write_bytes(pickle.dumps(self.cache,pickle.HIGHEST_PROTOCOL))
    def rerank(self,query,docs,k):
        docs=[d for d in docs if d.get("image_path")]; self.encode_missing(docs)
        if not docs:return []
        text_limit=int(getattr(self.model.config.text_config,"max_position_embeddings",77))
        x=self.processor(text=[query],return_tensors="pt",padding=True,truncation=True,max_length=text_limit).to(self.model.device)
        with torch.inference_mode():q=self.unwrap_features(self.model.get_text_features(**x)); q=q/q.norm(dim=-1,keepdim=True)
        q=q[0].cpu().numpy(); del x; out=[]
        for d in docs:out.append({**d,"clip_score":float(np.dot(q,self.cache[d["doc_id"]])),"final_score":float(np.dot(q,self.cache[d["doc_id"]]))})
        return sorted(out,key=lambda x:x["final_score"],reverse=True)[:k]
CLIP=ClipReranker()

In [ ]:
# Cell 12 - original-table graph and bounded cross-modal expansion
def norm_entity(x): return re.sub(r"\s+"," ",str(x).strip().lower())
def cell_entities(cell:Any)->List[str]:
    values=[cell_text(cell)]
    if isinstance(cell,dict):
        for key in ["links","entities","wiki_links"]:
            for link in cell.get(key,[]) or []:
                if isinstance(link,dict):values.extend([link.get("title",""),link.get("text","")])
                else:values.append(link)
    return list(dict.fromkeys("e::"+norm_entity(v) for v in values if 1<len(norm_entity(v))<=100 and not norm_entity(v).replace(".","").isdigit()))
def build_graph():
    paths,found,root=artifact_group([("graph","cache","mmqa_graph_v2.pkl")]); path=paths["graph"]; fp="v2-"+fingerprint(CORPUS)
    if found:
        payload=pickle.loads(path.read_bytes())
        if payload["fingerprint"]==fp:return payload["graph"]
        path=artifact_output_group([("graph","cache","mmqa_graph_v2.pkl")])["graph"]
    g=nx.Graph()
    for modality,raws in [("text",texts_raw),("image",images_raw)]:
        for raw in raws:
            did=str(raw["id"]); dn=f"d::{modality}::{did}"; title=norm_entity(raw.get("title",""))
            g.add_node(dn,kind="document",doc_id=did,modality=modality)
            if title:g.add_edge(dn,"e::"+title,kind="title")
    for raw in tables_raw:
        did=str(raw["id"]); dn=f"d::table::{did}"; title=norm_entity(raw.get("title",raw.get("name","")))
        g.add_node(dn,kind="document",doc_id=did,modality="table")
        if title:g.add_edge(dn,"e::"+title,kind="title")
        table=raw.get("table",raw); rows=table.get("table_rows",table.get("rows",[]))
        for row in rows:
            entities=list(dict.fromkeys(e for cell in row for e in cell_entities(cell)))[:20]
            for entity in entities:g.add_edge(dn,entity,kind="cell")
            for i,a in enumerate(entities):
                for b in entities[i+1:i+4]:g.add_edge(a,b,kind="row")
    path.write_bytes(pickle.dumps({"fingerprint":fp,"graph":g},pickle.HIGHEST_PROTOCOL)); return g
GRAPH=build_graph() if ENABLE_GRAPH else None
class WebRetriever:
    """Extension point only; web evidence is forbidden in official MMQA experiments."""
    def search(self,query:str,k:int=5):
        if not ENABLE_WEB: raise RuntimeError("Web retrieval is disabled for reproducibility and leakage control")
        raise NotImplementedError("Provide a versioned, auditable backend before enabling non-benchmark experiments")
def representative_record(modality:str,parent_id:str)->Optional[dict]:
    records=PARENT_LOOKUP.get((modality,parent_id),[])
    return records[0] if records else None
def cross_modal_graph_expand(seed_candidates:Dict[str,List[dict]])->Tuple[List[dict],List[dict]]:
    if not GRAPH:return [],[]
    seeds=sorted([d for xs in seed_candidates.values() for d in xs],key=lambda x:x.get("final_score",0),reverse=True)[:GRAPH_SEED_K]
    found={}; provenance=[]; seed_ids={(d["modality"],d["parent_doc_id"]) for d in seeds}
    for seed_rank,seed in enumerate(seeds,1):
        start=f'd::{seed["modality"]}::{seed["parent_doc_id"]}'
        if start not in GRAPH:continue
        queue=[(start,0,[start])]; visited={start}
        while queue:
            node,hop,path=queue.pop(0)
            if hop>=2:continue
            neighbors=sorted(GRAPH.neighbors(node),key=str)[:GRAPH_MAX_DEGREE]
            for nxt in neighbors:
                if nxt in visited:continue
                visited.add(nxt); new_path=path+[nxt]; new_hop=hop+1
                attrs=GRAPH.nodes[nxt]
                if attrs.get("kind")=="document":
                    key=(attrs["modality"],attrs["doc_id"])
                    if key not in seed_ids:
                        score=1/((RRF_K+seed_rank)*(new_hop+1)); old=found.get(key)
                        rec=representative_record(*key)
                        if rec and (old is None or score>old["graph_score"]):
                            found[key]={**rec,"dense_rank":None,"dense_score":None,"bm25_rank":None,"bm25_score":None,"hybrid_rrf_score":0.0,"graph_score":score,"final_score":score,"retrieved_by_subquery":seed.get("retrieved_by_subquery",[])}
                        if len(provenance)<GRAPH_MAX_EXPANDED_DOCS*4:
                            provenance.append({"seed_doc":seed["parent_doc_id"],"seed_modality":seed["modality"],"entity_path":new_path[1:-1],"expanded_doc":attrs["doc_id"],"expanded_modality":attrs["modality"],"hop_count":new_hop,"graph_score":score})
                queue.append((nxt,new_hop,new_path))
    ranked=sorted(found.values(),key=lambda x:x["graph_score"],reverse=True)[:GRAPH_MAX_EXPANDED_DOCS]
    for rank,d in enumerate(ranked,1):d["graph_rank"]=rank
    return ranked,provenance

In [ ]:
# Cell 13 - deterministic question-only planner for retrieval-only experiments
def question_only_plan(question:str)->dict:
    q=question.lower(); confidence={"text":0.50,"table":0.45,"image":0.20}
    if re.search(r"\b(color|colour|logo|pictured|shown|image|photo|looks?|wearing|flag|shape)\b",q):confidence["image"]=0.88
    if re.search(r"\b(how many|number|year|when|date|rank|score|record|role|list|first|last|most|least)\b",q):confidence["table"]=0.78
    if re.search(r"\b(according|described|stated|why|who|biography|history|where)\b",q):confidence["text"]=0.72
    relational=bool(re.search(r"\b(where|which|whose|that|who)\b.*\b(where|which|what|when|color|colour|year)\b",q))
    if relational:
        confidence["table"]=max(confidence["table"],0.70); confidence["text"]=max(confidence["text"],0.60)
    available=["text","table"]+(["image"] if IMAGE_PATHS else [])
    selected=[m for m in available if confidence[m]>=PLANNER_MODALITY_THRESHOLD]
    if not selected:selected=[max(available,key=lambda m:confidence[m])]
    routing_confidence=max(confidence[m] for m in selected)
    return {"needs_decomposition":False,"subqueries":[question],"needed_modalities":selected,"modality_confidence":confidence,"routing_confidence":routing_confidence,"needs_relational_search":relational,"planner_backend":"question_only_rules"}
def fallback_plan(question):return question_only_plan(question)

In [ ]:
# Cell 14 - retrieval memory checkpoint
if CLIP.model is not None: CLIP.model.to("cpu")
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()
gpu_memory("retrieval stack ready:")

In [ ]:
# Cell 15 - memory-bounded Qwen load, then safe small-encoder placement
QWEN_CALL_COUNTER=defaultdict(int); FORMAT_REPAIR_CALL_COUNTER=defaultdict(int); model=None; processor=None
from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration
from qwen_vl_utils import process_vision_info
gpu_memory("before Qwen:")
processor=AutoProcessor.from_pretrained(MODEL_NAME,min_pixels=256*28*28,max_pixels=384*28*28,use_fast=IMAGE_PROCESSOR_USE_FAST)
bnb=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
max_memory={i:"10GiB" for i in range(torch.cuda.device_count())}; max_memory["cpu"]="24GiB"
model=Qwen2_5_VLForConditionalGeneration.from_pretrained(MODEL_NAME,quantization_config=bnb,device_map="balanced",max_memory=max_memory,
    offload_folder=str(DIRS["offload"]),offload_state_dict=True,low_cpu_mem_usage=True,attn_implementation="sdpa").eval()
assert model is not None and processor is not None

if torch.cuda.is_available():
    free0,_=torch.cuda.mem_get_info(0)
    if free0/2**30>=BGE_GPU_MIN_FREE_GIB:
        QUERY_ENCODER.to("cuda:0"); BGE_DEVICE="cuda:0"
        for index in [TEXT_INDEX,TABLE_INDEX,IMAGE_TITLE_INDEX]:index.encoder=QUERY_ENCODER
    free0,_=torch.cuda.mem_get_info(0)
    if free0/2**30>=CLIP_GPU_MIN_FREE_GIB:CLIP_DEVICE="cuda:0"
print("QWEN LOADED; BGE device =",BGE_DEVICE,"; CLIP requested device =",CLIP_DEVICE)
print("MAMMQA: do_sample =",MAMMQA_DO_SAMPLE,"temperature =",MAMMQA_TEMPERATURE,"top_p =",MAMMQA_TOP_P)
print("stage_max_new_tokens =",STAGE_MAX_NEW_TOKENS,"repair_stages =",REPAIR_STAGES)
gpu_memory("post-placement:")

In [ ]:
# Cell 16 - compact parsers, protected evidence budgeting, aggregator-only repair and OOM recovery
def append_jsonl_log(path,row):
    with Path(path).open("a",encoding="utf-8") as f:
        f.write(json.dumps(row,ensure_ascii=False,default=str)+"\n"); f.flush(); os.fsync(f.fileno())

def clip01(x,default=0.0):
    try:return max(0.0,min(1.0,float(x)))
    except:return default

def stable_call_seed(qid,stage,retry_count):return (SEED+int(hashlib.sha256(f"{qid}:{stage}:{retry_count}".encode()).hexdigest()[:8],16))%(2**31-1)
def evidence_id(item):return str(item.get("evidence_id",item.get("doc_id","unknown")))
def public_evidence(item):return {k:item.get(k) for k in ["evidence_id","modality","content"] if item.get(k) is not None}

def build_messages(system,protected,evidence):
    content=[]
    for item in evidence:
        if item.get("modality")=="image" and item.get("image_path") and Path(item["image_path"]).is_file():content.append({"type":"image","image":f'file://{item["image_path"]}'})
    content.append({"type":"text","text":json.dumps({"protected":protected,"evidence":[public_evidence(x) for x in evidence]},ensure_ascii=False)})
    return [{"role":"system","content":system},{"role":"user","content":content}]

def processor_inputs(messages):
    prompt=processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True); images,videos=process_vision_info(messages)
    kwargs={"text":[prompt],"return_tensors":"pt","padding":True,"truncation":False}
    if images:kwargs.update(images=images,videos=videos)
    inputs=processor(**kwargs); return inputs,int(inputs["input_ids"].shape[-1]),len(images or [])

def shorten_lowest_ranked_text_evidence(evidence):
    for index in sorted(range(len(evidence)),key=lambda i:(evidence[i].get("rank",0),i),reverse=True):
        content=evidence[index].get("content")
        if isinstance(content,str) and len(content)>500:
            new=copy.deepcopy(evidence); new[index]["content"]=content[:max(500,len(content)//2)]+"\n[shortened]"; return new,True
    return evidence,False

def prepare_bounded_inputs(system,protected,evidence,max_budget=MAX_INPUT_TOKENS):
    work=copy.deepcopy(evidence); removed=[]; trimmed=False
    while True:
        messages=build_messages(system,protected,work); inputs,count,image_count=processor_inputs(messages)
        if count<=max_budget:return inputs,count,image_count,removed,trimmed
        del inputs
        shortened,did=shorten_lowest_ranked_text_evidence(work)
        if did:work=shortened; trimmed=True; continue
        if work:
            index=max(range(len(work)),key=lambda i:(work[i].get("rank",0),i)); removed.append(evidence_id(work[index])); work=work[:index]+work[index+1:]; trimmed=True; continue
        raise RuntimeError(f"PROTECTED_PROMPT_TOO_LARGE: {count}>{max_budget}")

def tag_value(text,name):
    closed=re.search(fr"<{name}>\s*(.*?)\s*</{name}>",str(text),re.I|re.S)
    if closed:return closed.group(1).strip()
    partial=re.search(fr"<{name}>\s*(.*?)(?=<[a-z_]+>|$)",str(text),re.I|re.S)
    return partial.group(1).strip() if partial else ""

def parse_tagged_agent(raw,stage):
    insights=tag_value(raw,"insight") or tag_value(raw,"insights"); answer=tag_value(raw,"answer"); subanswer=tag_value(raw,"subanswer"); ids=tag_value(raw,"evidence_ids")
    if not insights or not answer:return None
    parsed={"insights":insights,"candidate_answer":answer,"subanswer":subanswer,
            "supporting_doc_ids":[x.strip() for x in re.split(r"[,;]",ids) if x.strip()]}
    if stage.startswith("stage1_"):
        expected=stage.split("_",1)[1]; modality=tag_value(raw,"modality").lower()
        if modality!=expected:return None
        parsed["modality"]=modality
    return parsed

def parse_compact_planner(raw):
    modalities_match=re.search(r"^\s*MODALITIES\s*:\s*([^\n]+)",str(raw),re.I|re.M)
    if not modalities_match:return None
    modalities=[m.lower() for m in re.findall(r"text|table|image",modalities_match.group(1),re.I)]
    if not modalities:return None
    queries=[]
    for value in re.findall(r"^\s*QUERY(?:[123])?\s*:\s*([^\n]+)",str(raw),re.I|re.M):
        value=value.strip()
        if value and value.lower() not in {"none","n/a"}:queries.append(value)
    weights={m:1/3 for m in ["text","table","image"]}; wm=re.search(r"^\s*WEIGHTS\s*:\s*([^\n]+)",str(raw),re.I|re.M)
    if wm:
        found={m.lower():float(v) for m,v in re.findall(r"(text|table|image)\s*=\s*(0(?:\.\d+)?|1(?:\.0+)?)",wm.group(1),re.I)}
        if found:
            total=sum(found.values()) or 1.0; weights={m:found.get(m,0.0)/total for m in ["text","table","image"]}
    return {"needed_modalities":list(dict.fromkeys(modalities)),"subqueries":queries[:3],"modality_confidence":weights}

def parse_compact_verifier(raw):
    def line(label):
        match=re.search(fr"^\s*{label}\s*:\s*([^\n]*)",str(raw),re.I|re.M); return match.group(1).strip() if match else ""
    status=line("STATUS").lower(); best=line("BEST_ANSWER"); ids=line("SUPPORT_IDS")
    mod=line("RETRY_MODALITY").lower(); query=line("RETRY_QUERY")
    if status not in {"supported","insufficient","conflict"}:return None
    if status=="supported" and (not best or best.upper()=="NONE"):return None
    if mod not in {"text","table","image","none"}:mod="none"
    return {"status":status,"best_answer":"" if best.upper()=="NONE" else best,
            "supporting_doc_ids":[x.strip() for x in re.split(r"[,;]",ids) if x.strip() and x.strip().upper()!="NONE"],
            "retry_modality":mod,"retry_query":"" if query.upper()=="NONE" else query}

def parse_compact_aggregator(raw):
    answer=tag_value(raw,"answer")
    if not answer:
        match=re.search(r"^\s*ANSWER\s*:\s*([^\n]+)",str(raw),re.I|re.M); answer=match.group(1).strip() if match else ""
    if not answer:return None
    return {"answer":answer,"status":"abstain" if answer.strip().upper()=="ABSTAIN" else "answered"}

def stage_budget(stage):
    key="stage1" if stage.startswith("stage1_") else ("stage2" if stage.startswith("stage2_") else ("verifier" if stage.startswith("verifier") else stage.replace("_format_repair","")))
    return STAGE_MAX_NEW_TOKENS[key]

def generate_once(system,protected,evidence,qid,stage,do_sample,retry_count=0):
    max_new_tokens=stage_budget(stage); inputs,input_count,image_count,removed,trimmed=prepare_bounded_inputs(system,protected,evidence)
    seed=stable_call_seed(qid,stage,retry_count); random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
    inputs=inputs.to(model.device); kwargs={"max_new_tokens":max_new_tokens,"do_sample":do_sample}
    if do_sample:kwargs.update(temperature=MAMMQA_TEMPERATURE,top_p=MAMMQA_TOP_P)
    started=time.perf_counter(); oom_retried=False
    print(f"[QWEN START] qid={qid} stage={stage} input={input_count} images={image_count} max_new={max_new_tokens}",flush=True)
    try:
        QWEN_CALL_COUNTER[qid]+=1
        with torch.inference_mode():ids=model.generate(**inputs,**kwargs)
    except torch.cuda.OutOfMemoryError:
        oom_retried=True; gc.collect(); torch.cuda.empty_cache(); print(f"[OOM RETRY] qid={qid} stage={stage}",flush=True)
        try:
            QWEN_CALL_COUNTER[qid]+=1
            with torch.inference_mode():ids=model.generate(**inputs,**kwargs)
        except torch.cuda.OutOfMemoryError:
            append_jsonl_log(GENERATION_DIAGNOSTICS_PATH,{"qid":qid,"stage":stage,"oom":True,"oom_retry_failed":True,"time":time.time()}); del inputs; raise
    output_ids=ids[:,inputs.input_ids.shape[1]:]; raw=processor.batch_decode(output_ids,skip_special_tokens=True)[0].strip()
    meta={"qid":qid,"stage":stage,"input_token_count":input_count,"max_input_budget":MAX_INPUT_TOKENS,"image_count":image_count,
          "output_token_count":int(output_ids.shape[-1]),"max_new_tokens":max_new_tokens,"retry_count":retry_count,
          "generation_time":time.perf_counter()-started,"context_trimmed":trimmed,"removed_evidence_ids":removed,
          "protected_prompt_truncated":False,"seed":seed,"oom_retried":oom_retried,"hit_max_new_tokens":int(output_ids.shape[-1])>=max_new_tokens}
    print(f"[QWEN DONE] qid={qid} stage={stage} output={meta['output_token_count']} seconds={meta['generation_time']:.1f}",flush=True)
    del inputs,ids,output_ids
    return raw,meta

REPAIR_PROMPT="""Reformat the previous output only. Do not add new evidence or facts. Return exactly <answer>short final answer OR ABSTAIN</answer>."""

def stage_call(system,protected,evidence,qid,stage,do_sample,parser,allow_repair=False):
    assert allow_repair==(stage in REPAIR_STAGES)
    raw,meta=generate_once(system,protected,evidence,qid,stage,do_sample,0); parsed=parser(raw); ok=parsed is not None
    meta["parse_success"]=ok; append_jsonl_log(GENERATION_DIAGNOSTICS_PATH,meta)
    if meta["context_trimmed"]:append_jsonl_log(TRUNCATION_DIAGNOSTICS_PATH,meta)
    repair_raw=None; diagnostics=[meta]
    if not ok and allow_repair:
        FORMAT_REPAIR_CALL_COUNTER[qid]+=1
        repair_raw,rmeta=generate_once(REPAIR_PROMPT,{"previous_output":raw[:2500]},[],qid,stage+"_format_repair",False,1)
        parsed=parser(repair_raw); ok=parsed is not None; rmeta["parse_success"]=ok; append_jsonl_log(GENERATION_DIAGNOSTICS_PATH,rmeta); diagnostics.append(rmeta)
    print(f"[PARSE] qid={qid} stage={stage} success={ok} repair={repair_raw is not None}",flush=True)
    return {"raw_output":raw,"repair_raw_output":repair_raw,"parsed_output":parsed if ok else None,"parse_success":ok,
            "retry_count":int(repair_raw is not None),"generation_schema_failure":not ok,"generation_diagnostics":diagnostics}

def compact_or_raw(envelope,max_chars=1600):
    if envelope and envelope.get("parse_success"):return envelope["parsed_output"]
    return {"schema_failure":True,"raw_output":str((envelope or {}).get("raw_output",""))[:max_chars]}

def canonical_answer(value):
    answer=str(value or "").strip().strip('"\''); low=answer.lower().strip(" .")
    if low in {"yes","yeah","true"}:return "yes"
    if low in {"no","false"}:return "no"
    if re.fullmatch(r"[-+]?\d{1,3}(?:,\d{3})+(?:\.\d+)?",answer):return answer.replace(",","")
    return answer

def parse_final_answer(envelope):
    parsed=(envelope or {}).get("parsed_output") if (envelope or {}).get("parse_success") else None
    if parsed and str(parsed.get("answer","")).strip():return canonical_answer(parsed["answer"]),True
    for text in [(envelope or {}).get("raw_output",""),(envelope or {}).get("repair_raw_output","")]:
        answer=tag_value(text,"answer")
        if answer:return canonical_answer(answer),True
    return "",False

In [ ]:
# Cell 17 - compact one-call planner with deterministic safe fallback
PLANNER_PROMPT="""Plan retrieval from the question only. Return at most these lines, with no explanation:
MODALITIES: comma-separated text,table,image
WEIGHTS: text=0.xx,table=0.xx,image=0.xx
QUERY1: short retrieval query
QUERY2: optional short retrieval query
QUERY3: optional short retrieval query"""

def normalize_subquery(value):
    if isinstance(value,str):return value.strip()
    if isinstance(value,dict):
        for key in ["query","subquery","question","text"]:
            if isinstance(value.get(key),str) and value[key].strip():return value[key].strip()
    return ""
def assert_clean_retrieval_query(query):
    stripped=str(query).lstrip(); assert not stripped.startswith(("{'query':",'{"query":')); assert not re.search(r"['\"]modality_confidence['\"]\s*:",str(query))
def build_query_records(question,subqueries):
    records=[]; seen=set(); original=str(question).strip()
    for value,source in [(original,"original_question")]+[(x,"decomposed_subquery") for x in subqueries]:
        text=original if source=="original_question" else normalize_subquery(value); key=" ".join(text.split()).casefold()
        if text and key not in seen:assert_clean_retrieval_query(text); records.append({"text":text,"source":source}); seen.add(key)
    assert records[0]=={"text":original,"source":"original_question"}; return records

def safe_planner_fallback(question,reason):
    mods=["text","table"]+(["image"] if IMAGE_PATHS else []); weights={m:1/len(mods) for m in mods}
    return {"needed_modalities":mods,"modality_confidence":{m:weights.get(m,0.0) for m in ["text","table","image"]},
            "routing_confidence":max(weights.values()),"normalized_subqueries":[],"retrieval_queries":build_query_records(question,[]),
            "planner_parse_failure":True,"fallback_reason":reason}

def fallback_plan(question,reason="planner_disabled"):
    plan=safe_planner_fallback(question,reason); plan.update(raw_planner_output=None,parsed_planner_output=None,planner_parse_success=True,planner_schema_failure=False,planner_parse_failure=False,planner_generation=None)
    return plan

def plan_query(q):
    assert_safe_query(q); started=time.perf_counter()
    envelope=stage_call(PLANNER_PROMPT,{"qid":q.qid,"question":q.question},[],q.qid,"planner",PLANNER_DO_SAMPLE,parse_compact_planner,False)
    parsed=envelope.get("parsed_output")
    if parsed:
        normalized=[normalize_subquery(x) for x in parsed.get("subqueries",[])]; normalized=[x for x in normalized if x][:3]
        mods=[m for m in parsed["needed_modalities"] if m in {"text","table","image"} and (m!="image" or IMAGE_PATHS)]
        if not mods:mods=["text","table"]+(["image"] if IMAGE_PATHS else [])
        conf={m:clip01(parsed.get("modality_confidence",{}).get(m),0.0) for m in ["text","table","image"]}
        plan={"needed_modalities":mods,"modality_confidence":conf,"routing_confidence":max(conf.values()),"normalized_subqueries":normalized,
              "retrieval_queries":build_query_records(q.question,normalized),"planner_parse_failure":False}
    else:plan=safe_planner_fallback(q.question,"planner_parse_failure")
    plan.update(raw_planner_output=envelope["raw_output"],parsed_planner_output=parsed,planner_parse_success=envelope["parse_success"],
                planner_schema_failure=envelope["generation_schema_failure"],planner_generation=envelope,planner_backend="qwen_compact",
                planner_latency_seconds=time.perf_counter()-started)
    return plan

In [ ]:
# Cell 18 - soft-priority routing, full candidate retrieval and separate context selection
def ensure_encoder():
    assert QUERY_ENCODER is not None,"Persistent query encoder was not initialized"; return QUERY_ENCODER

def route_budgets(plan,router_mode):
    assert router_mode in {"hard","soft","all"}
    available=["text","table"]+(["image"] if IMAGE_PATHS else [])
    normal={"text":TEXT_CONTEXT_K,"table":TABLE_CONTEXT_K,"image":IMAGE_CONTEXT_K}
    fallback={"text":FALLBACK_TEXT_K,"table":FALLBACK_TABLE_K,"image":FALLBACK_IMAGE_K}
    selected=set(plan.get("needed_modalities",[]))&set(available)
    if not selected:selected={max(available,key=lambda m:plan.get("modality_confidence",{}).get(m,0))}
    low_conf=float(plan.get("routing_confidence",0))<ROUTER_CONFIDENCE_THRESHOLD
    # Soft routing is prioritization only: every modality is searched at the full candidate depth.
    if router_mode in {"soft","all"}:
        candidates={m:max(RETRIEVAL_CANDIDATE_K,MIN_METRIC_CANDIDATE_K) for m in available}
        if router_mode=="all" or low_conf:contexts={m:normal[m] for m in available}
        else:contexts={m:(normal[m] if m in selected else fallback[m]) for m in available}
        return candidates,contexts
    return ({m:max(RETRIEVAL_CANDIDATE_K,MIN_METRIC_CANDIDATE_K) for m in selected},{m:normal[m] for m in selected})

def merge_subquery_rankings(groups,k):
    merged={}
    for group in groups:
        for rank,d in enumerate(group,1):
            key=d["doc_id"]; x=merged.setdefault(key,{**d,"subquery_rrf_score":0.0,"retrieved_by_subquery":[],"query_provenance":[]})
            x["subquery_rrf_score"]+=1/(RRF_K+rank)
            x["retrieved_by_subquery"]=list(dict.fromkeys(x["retrieved_by_subquery"]+d.get("retrieved_by_subquery",[])))
            x["query_provenance"]+= [p for p in d.get("query_provenance",[]) if p not in x["query_provenance"]]
            for field in ["dense_score","bm25_score","hybrid_rrf_score"]:
                values=[v for v in [x.get(field),d.get(field)] if v is not None]; x[field]=max(values) if values else None
    for x in merged.values():x["final_score"]=x["subquery_rrf_score"]
    return sorted(merged.values(),key=lambda x:x["final_score"],reverse=True)[:k]

def aggregate_parent_candidates(items,k):
    groups=defaultdict(list)
    for d in items:groups[d["parent_doc_id"]].append(d)
    parents=[]
    for parent_id,chunks in groups.items():
        chunks=sorted(chunks,key=lambda x:x.get("final_score",0),reverse=True); best=dict(chunks[0])
        best["matched_chunks"]=[{"chunk_id":x["doc_id"],"row_start":x.get("row_start"),"row_end":x.get("row_end"),"score":x.get("final_score",0)} for x in chunks[:5]]
        best["retrieved_by_subquery"]=list(dict.fromkeys(q for x in chunks for q in x.get("retrieved_by_subquery",[])))
        best["query_provenance"]=[p for x in chunks for p in x.get("query_provenance",[])]
        parents.append(best)
    return sorted(parents,key=lambda x:x.get("final_score",0),reverse=True)[:k]

def fuse_graph(base,expanded,k_by_mod):
    out={m:list(base.get(m,[])) for m in ["text","table","image"]}; graph_by_mod=defaultdict(list)
    for d in expanded:graph_by_mod[d["modality"]].append(d)
    for mod in out:
        base_rank={d["parent_doc_id"]:i+1 for i,d in enumerate(out[mod])}; graph_rank={d["parent_doc_id"]:i+1 for i,d in enumerate(graph_by_mod[mod])}
        records={d["parent_doc_id"]:dict(d) for d in out[mod]}
        for d in graph_by_mod[mod]:records.setdefault(d["parent_doc_id"],dict(d))
        fused=[]
        for did,d in records.items():
            d["graph_rank"]=graph_rank.get(did); d["graph_score"]=next((x.get("graph_score",0) for x in graph_by_mod[mod] if x["parent_doc_id"]==did),0)
            d["final_score"]=(1/(RRF_K+base_rank[did]) if did in base_rank else 0)+(1/(RRF_K+graph_rank[did]) if did in graph_rank else 0)
            if d["graph_rank"]:d.setdefault("query_provenance",[]).append({"query_text":"GRAPH_EXPANSION","query_source":"graph_expansion","rank":d["graph_rank"],"score":d["graph_score"],"method":"graph"})
            fused.append(d)
        out[mod]=sorted(fused,key=lambda x:x["final_score"],reverse=True)[:k_by_mod.get(mod,RETRIEVAL_CANDIDATE_K)]
    return out

def candidate_terms(doc):
    text=" ".join(str(doc.get(k,"")) for k in ["title","retrieval_text","reasoning_content","raw_content"])
    return set(re.findall(r"[a-z0-9]+",text.lower()))

def redundancy(a,b):
    x,y=candidate_terms(a),candidate_terms(b)
    return len(x&y)/max(1,len(x|y))

def query_hits(doc):
    return [(str(p.get("query_text","")),str(p.get("query_source",""))) for p in doc.get("query_provenance",[]) if p.get("query_text")]

def select_modality_context(modality,pool,base_k,max_k,plan,feedback_doc_ids=None):
    feedback_doc_ids=set(map(str,feedback_doc_ids or [])); pool=[dict(d) for d in pool]
    for rank,d in enumerate(pool,1):d.setdefault("_candidate_rank",rank)
    query_records=plan.get("retrieval_queries",[]); original=next((q["text"] for q in query_records if q.get("source")=="original_question"),"")
    subqueries=[q["text"] for q in query_records if q.get("source")=="decomposed_subquery"]
    needed=modality in set(plan.get("needed_modalities",[])); confidence=float(plan.get("modality_confidence",{}).get(modality,0))
    top=pool[:max(1,base_k)]; duplicate_top=any(redundancy(a,b)>=CONTEXT_DUPLICATE_THRESHOLD for i,a in enumerate(top) for b in top[i+1:])
    query_space={q for d in pool for q,_ in query_hits(d)}; uncovered_possible=sum(q in query_space for q in subqueries)>max(0,base_k-1)
    target=base_k
    if needed and target<max_k and (uncovered_possible or duplicate_top or (subqueries and confidence>=0.55)):target+=1
    target=min(target,max_k,len(pool)); selected=[]; covered=set(); diagnostics=[]

    def add(doc,reason,score):
        hits=query_hits(doc); source=hits[0][0] if hits else ("GRAPH_EXPANSION" if doc.get("graph_score") else "")
        item=dict(doc); item.update(selected_rank=len(selected)+1,candidate_rank=doc["_candidate_rank"],source_query=source,selection_reason=reason,selection_score=float(score))
        selected.append(item); covered.update(q for q,_ in hits)
        diagnostics.append({"doc_id":str(doc["parent_doc_id"]),"modality":modality,"selected_rank":len(selected),"candidate_rank":doc["_candidate_rank"],
                            "retrieval_score":float(doc.get("final_score",0)),"source_query":source,"selection_reason":reason,"selection_score":float(score)})

    original_candidates=[d for d in pool if any(source=="original_question" or query==original for query,source in query_hits(d))]
    if target and original_candidates:add(original_candidates[0],"original_question_top",2.0)
    while len(selected)<target:
        best=None
        for d in pool:
            if any(str(x["parent_doc_id"])==str(d["parent_doc_id"]) for x in selected):continue
            hits=query_hits(d); new_queries={q for q,_ in hits if q not in covered}; rel=1/max(1,d["_candidate_rank"])
            red=max([redundancy(d,x) for x in selected] or [0.0]); graph=1.0 if d.get("graph_score",0)>0 else 0.0
            is_feedback=str(d["parent_doc_id"]) in feedback_doc_ids
            score=(CONTEXT_SELECTION_WEIGHTS["relevance"]*rel+
                   CONTEXT_SELECTION_WEIGHTS["uncovered_query"]*bool(new_queries)+
                   CONTEXT_SELECTION_WEIGHTS["modality_need"]*needed+
                   CONTEXT_SELECTION_WEIGHTS["graph"]*graph+
                   CONTEXT_SELECTION_WEIGHTS["feedback"]*is_feedback-
                   CONTEXT_SELECTION_WEIGHTS["redundancy"]*red)
            reason=("feedback_added" if is_feedback else "subquery_coverage" if new_queries else "graph_expansion" if graph else
                    "modality_coverage" if needed and not selected else "diversity_fill" if red<CONTEXT_DUPLICATE_THRESHOLD else "high_fused_score")
            candidate=(score,-d["_candidate_rank"],d,reason)
            if best is None or candidate[:2]>best[:2]:best=candidate
        if best is None:break
        add(best[2],best[3],best[0])
    return selected,diagnostics,{"base_k":base_k,"selected_k":len(selected),"max_k":max_k,"adaptive_expansion":len(selected)>base_k,
                                 "covered_queries":sorted(covered),"duplicate_top_detected":duplicate_top}

def select_context(question,candidates,context_budgets,plan,feedback_doc_ids=None):
    context={m:[] for m in ["text","table","image"]}; diagnostics=[]; stats={}
    max_k={"text":TEXT_CONTEXT_K_MAX,"table":TABLE_CONTEXT_K_MAX,"image":IMAGE_CONTEXT_K_MAX}
    for mod,k in context_budgets.items():
        pool=list(candidates.get(mod,[]))
        if mod=="image" and pool:
            ranks={str(d["parent_doc_id"]):i+1 for i,d in enumerate(pool)}
            pool=CLIP.rerank(question,[d for d in pool if d.get("image_path")],len(pool))
            for d in pool:d["_candidate_rank"]=ranks.get(str(d["parent_doc_id"]),len(pool)+1)
        chosen,rows,mod_stats=select_modality_context(mod,pool,k,max_k[mod],plan,feedback_doc_ids)
        context[mod]=chosen; diagnostics.extend(rows); stats[mod]=mod_stats
    return context,diagnostics,stats

def retrieve_evidence(q,plan,retrieval_mode=RETRIEVAL_MODE,router_mode=ROUTER_MODE,use_graph=True,fixed_all=False):
    assert_safe_query(q); ensure_encoder(); started=time.perf_counter()
    if fixed_all:router_mode="all"
    candidate_budgets,context_budgets=route_budgets(plan,router_mode)
    query_records=plan.get("retrieval_queries") or build_query_records(q.question,plan.get("normalized_subqueries",[]))
    assert query_records[0]=={"text":q.question,"source":"original_question"}
    candidates={m:[] for m in ["text","table","image"]}
    for mod,k in candidate_budgets.items():
        groups=[search_modality(mod,qr["text"],max(k,RETRIEVAL_CANDIDATE_K),retrieval_mode,qr["text"],qr["source"]) for qr in query_records]
        chunks=merge_subquery_rankings(groups,max(k*3,RETRIEVAL_CANDIDATE_K)); candidates[mod]=aggregate_parent_candidates(chunks,k)
        assert len(candidates[mod])>=min(MIN_METRIC_CANDIDATE_K,len(BY_MODALITY[mod]))
    graph_provenance=[]
    if use_graph:
        expanded,graph_provenance=cross_modal_graph_expand(candidates)
        limits={m:max(candidate_budgets.get(m,0),MIN_METRIC_CANDIDATE_K) for m in candidates}
        candidates=fuse_graph(candidates,expanded,limits)
    context,selection_diagnostics,selection_stats=select_context(q.question,candidates,context_budgets,plan)
    priorities=sorted(({"modality":m,"confidence":float(plan.get("modality_confidence",{}).get(m,0)),"candidate_budget":candidate_budgets[m],"context_budget":context_budgets.get(m,0)} for m in candidate_budgets),key=lambda x:(-x["confidence"],x["modality"]))
    diagnostic={"qid":q.qid,"question":q.question,"raw_planner_output":plan.get("raw_planner_output"),
                "parsed_planner_output":plan.get("parsed_planner_output"),"normalized_subqueries":plan.get("normalized_subqueries",[]),
                "retrieval_queries":query_records,"modality_confidence":plan.get("modality_confidence",{}),
                "modalities_searched":list(candidate_budgets),"candidate_counts":{m:len(xs) for m,xs in candidates.items()},
                "context_counts":{m:len(xs) for m,xs in context.items()},"context_selection":selection_diagnostics,"context_selection_stats":selection_stats}
    append_jsonl_log(PLANNER_DIAGNOSTICS_PATH,diagnostic)
    return {"retrieval_candidates":candidates,"context_evidence":context,"graph_provenance":graph_provenance,"context_selection_diagnostics":selection_diagnostics,"context_selection_stats":selection_stats,
            "candidate_ids":{m:[d["parent_doc_id"] for d in xs] for m,xs in candidates.items()},
            "context_ids":{m:[d["parent_doc_id"] for d in xs] for m,xs in context.items()},
            "routing":{"mode":router_mode,"retrieval_mode":retrieval_mode,"candidate_budgets":candidate_budgets,"context_budgets":context_budgets,"searched_modalities":list(candidate_budgets),"soft_router_priorities":priorities},
            "retrieval_latency_seconds":time.perf_counter()-started}

In [ ]:
# Cell 19 - compact Stage-I modality experts; no repair generation
EXPERT_PROMPT="""You are the modality-specific MAMMQA expert. Use only the supplied modality evidence.
Return exactly:
<modality>text|table|image</modality>
<insights>Maximum 2 concise sentences.</insights>
<answer>short candidate answer OR INSUFFICIENT</answer>
<subanswer>optional very short supporting result</subanswer>
<evidence_ids>comma-separated supplied IDs or NONE</evidence_ids>
No long explanation and no outside knowledge."""

def clean_evidence(items,modality,content_chars):
    clean=[]
    for rank,d in enumerate(items,1):
        body=d.get("reasoning_content",d.get("raw_content",""))
        if modality=="image":body="Official MultiModalQA image pixels are attached."
        elif not isinstance(body,str):body=json.dumps(body,ensure_ascii=False,default=str)
        clean.append({"evidence_id":str(d["parent_doc_id"]),"modality":modality,"content":body[:content_chars],
                      "image_path":d.get("image_path") if modality=="image" else None,"rank":rank})
    return clean

def run_modality_experts(q,evidence):
    assert_inference_boundary("run_modality_experts",q); out={}
    for mod in ["text","table","image"]:
        items=[d for d in evidence.get(mod,[]) if mod!="image" or d.get("image_path")]
        if not items:continue
        ev=clean_evidence(items,mod,STAGE1_EVIDENCE_CHARS); allowed={x["evidence_id"] for x in ev}
        envelope=stage_call(EXPERT_PROMPT,{"qid":q.qid,"question":q.question,"modality":mod},ev,q.qid,f"stage1_{mod}",MAMMQA_DO_SAMPLE,
                            lambda raw,s=f"stage1_{mod}":parse_tagged_agent(raw,s),False)
        parsed=envelope.get("parsed_output")
        if parsed:parsed["supporting_doc_ids"]=[x for x in parsed.get("supporting_doc_ids",[]) if x in allowed]
        out[mod]=envelope
    return out

In [ ]:
# Cell 20 - compact paper-style anchor synthesis; no repair generation
SYNTH_PROMPT="""Synthesize from the named anchor expert and raw evidence from other modalities only.
Return exactly:
<answer>short candidate answer OR INSUFFICIENT</answer>
<insight>one concise evidence-grounded sentence</insight>
<evidence_ids>comma-separated supplied IDs or NONE</evidence_ids>
No long explanation, chain-of-thought, or outside knowledge."""

def run_cross_modal_agents(q,experts,evidence):
    assert_inference_boundary("run_cross_modal_agents",q); out={}; allowed={str(d["parent_doc_id"]) for xs in evidence.values() for d in xs}
    for anchor,expert in experts.items():
        other=[]
        for mod in ["text","table","image"]:
            if mod!=anchor:other.extend(clean_evidence(evidence.get(mod,[]),mod,STAGE2_EVIDENCE_CHARS))
        protected={"qid":q.qid,"question":q.question,"anchor_modality":anchor,"anchor_expert":compact_or_raw(expert,1000)}
        envelope=stage_call(SYNTH_PROMPT,protected,other,q.qid,f"stage2_{anchor}",MAMMQA_DO_SAMPLE,
                            lambda raw,s=f"stage2_{anchor}":parse_tagged_agent(raw,s),False)
        parsed=envelope.get("parsed_output")
        if parsed:parsed["supporting_doc_ids"]=[x for x in parsed.get("supporting_doc_ids",[]) if x in allowed]
        out[anchor]=envelope
    return out

In [ ]:
# Cell 21 - evidence-aware verifier: concrete candidates, neutral INSUFFICIENT votes, deterministic decoding
VERIFY_PROMPT="""You are an evidence verifier. Decide whether available retrieved evidence supports a concrete candidate.
Do NOT require every modality or agent to answer. An agent output of INSUFFICIENT is neutral, not a contradiction.
SUPPORTED: at least one concrete candidate has direct support and no stronger contradictory evidence.
INSUFFICIENT: no concrete candidate has enough support yet.
CONFLICT: two or more different concrete answers have credible conflicting support.
Return compact lines only.
For supported:
STATUS: supported
BEST_ANSWER: short answer
SUPPORT_IDS: comma-separated IDs or NONE
For insufficient/conflict also add:
STATUS: insufficient | conflict
RETRY_MODALITY: text | table | image | none
RETRY_QUERY: short query or NONE"""

def normalize_consensus_answer(value):
    value=str(value or "").strip().strip('"\'').lower(); value="".join(" " if c in string.punctuation else c for c in value)
    return " ".join(value.split())

def synthesis_consensus(synth):
    answers=[]
    for modality,envelope in synth.items():
        parsed=envelope.get("parsed_output") if envelope.get("parse_success") else None
        answer=(parsed or {}).get("candidate_answer","")
        if not answer:answer=tag_value(envelope.get("raw_output",""),"answer")
        normalized=normalize_consensus_answer(answer)
        concrete=bool(normalized and normalized not in {"insufficient","abstain","none","unknown"})
        answers.append({"modality":modality,"answer":answer,"normalized_answer":normalized,"concrete":concrete,
                        "evidence_ids":list(map(str,(parsed or {}).get("supporting_doc_ids",[])))})
    groups=defaultdict(list)
    for row in answers:
        if row["concrete"]:groups[row["normalized_answer"]].append(row)
    best=max(groups.items(),key=lambda kv:(len(kv[1]),kv[0]),default=("",[]))
    return {"stage2_answers":answers,"concrete_answer_count":sum(x["concrete"] for x in answers),
            "normalized_answer_groups":{k:len(v) for k,v in groups.items()},
            "consensus_answer":best[1][0]["answer"] if best[1] else "","consensus_count":len(best[1]),
            "candidate_evidence_ids":sorted({i for x in answers for i in x["evidence_ids"]})}

def verifier_evidence(synth,evidence):
    cited=set(synthesis_consensus(synth)["candidate_evidence_ids"]); selected=[]
    for mod in ["text","table","image"]:
        docs=evidence.get(mod,[]); preferred=[d for d in docs if str(d["parent_doc_id"]) in cited]
        selected.extend(clean_evidence((preferred or docs)[:1],mod,VERIFIER_EVIDENCE_CHARS))
    return selected

def verify_answer(q,synth,evidence,consensus=None,stage_name="verifier"):
    assert_inference_boundary("verify_answer",q); consensus=consensus or synthesis_consensus(synth)
    protected={"qid":q.qid,"question":q.question,"candidate_syntheses":{m:compact_or_raw(x,700) for m,x in synth.items()},"consensus_diagnostics":consensus}
    envelope=stage_call(VERIFY_PROMPT,protected,verifier_evidence(synth,evidence),q.qid,stage_name,VERIFIER_DO_SAMPLE,parse_compact_verifier,False)
    parsed=envelope.get("parsed_output"); envelope["effective_status"]=parsed.get("status") if parsed else "verifier_error"
    envelope["verifier_error"]=not bool(parsed); envelope["verifier_parse_failure"]=not bool(parsed)
    return envelope

In [ ]:
# Cell 22 - executable one-round feedback, second cheap verifier and evidence-aware aggregation
AGG_PROMPT="""Resolve the synthesis results using the verifier decision.
ABSTAIN only when evidence genuinely remains insufficient or supported answers remain in conflict.
INSUFFICIENT outputs from other agents are neutral and do not invalidate a verifier-supported answer.
If verifier status is supported, normally preserve its BEST_ANSWER unless the supplied syntheses show a genuine contradiction.
Return exactly: <answer>short final answer OR ABSTAIN</answer>"""

def run_reasoning(q,evidence):
    t=time.perf_counter(); experts=run_modality_experts(q,evidence); stage1=time.perf_counter()-t
    t=time.perf_counter(); synth=run_cross_modal_agents(q,experts,evidence); stage2=time.perf_counter()-t
    return experts,synth,{"stage1_seconds":stage1,"stage2_seconds":stage2}

def aggregate_answer(q,synth,verification,consensus,sees_question=True):
    protected={"qid":q.qid,"syntheses":{m:compact_or_raw(x,700) for m,x in synth.items()},
               "verifier":compact_or_raw(verification,700),"consensus_diagnostics":consensus}
    if sees_question:protected["question"]=q.question
    return stage_call(AGG_PROMPT,protected,[],q.qid,"aggregator",MAMMQA_DO_SAMPLE,parse_compact_aggregator,True)

def merge_evidence(old,new):
    out={}
    for mod in ["text","table","image"]:
        vals={str(d["parent_doc_id"]):dict(d) for d in old.get(mod,[])}
        for d in new.get(mod,[]):
            did=str(d["parent_doc_id"])
            if did not in vals:vals[did]=dict(d); continue
            current=vals[did]; combined=current.get("query_provenance",[])+d.get("query_provenance",[]); unique={}
            for p in combined:unique[(str(p.get("query_text","")),str(p.get("query_source","")),int(p.get("rank",0)),str(p.get("method","")))]=p
            current["query_provenance"]=list(unique.values())
            for field in ["final_score","dense_score","bm25_score","hybrid_rrf_score","graph_score"]:
                values=[x for x in [current.get(field),d.get(field)] if x is not None]
                if values:current[field]=max(values)
        out[mod]=sorted(vals.values(),key=lambda d:(-float(d.get("final_score",0)),str(d["parent_doc_id"])))
    return out

def compact_pool(pool):
    keep=["parent_doc_id","doc_id","modality","final_score","dense_score","bm25_score","hybrid_rrf_score","graph_score","query_provenance",
          "candidate_rank","selected_rank","source_query","selection_reason","selection_score"]
    return {m:[{k:d.get(k) for k in keep if d.get(k) is not None} for d in docs] for m,docs in pool.items()}

def compact_round(round_no,query_records,bundle):
    return {"round":round_no,"query_records":query_records,"candidate_ids":bundle["candidate_ids"],"context_ids":bundle["context_ids"],
            "routing":bundle["routing"],"retrieval_latency_seconds":bundle["retrieval_latency_seconds"],
            "context_selection_diagnostics":bundle.get("context_selection_diagnostics",[])}

def deterministic_feedback_request(q,plan,synth,verifier_parsed):
    status=(verifier_parsed or {}).get("status")
    if status not in {"insufficient","conflict"}:return None
    available=["text","table"]+(["image"] if IMAGE_PATHS else [])
    supplied_mod=(verifier_parsed or {}).get("retry_modality"); supplied_query=str((verifier_parsed or {}).get("retry_query","")).strip()
    if supplied_mod in available and supplied_query:return {"modality":supplied_mod,"query":supplied_query,"source":"verifier"}
    consensus=synthesis_consensus(synth); insufficient=[x["modality"] for x in consensus["stage2_answers"] if not x["concrete"]]
    conflict_modalities=[x["modality"] for x in consensus["stage2_answers"] if x["concrete"]] if status=="conflict" else []
    choices=conflict_modalities or insufficient or available; weights=plan.get("modality_confidence",{})
    modality=supplied_mod if supplied_mod in available else max([m for m in choices if m in available] or available,key=lambda m:(float(weights.get(m,0)),m))
    queries=[x for x in plan.get("normalized_subqueries",[]) if str(x).strip()]; query=supplied_query or (queries[0] if queries else q.question)
    return {"modality":modality,"query":query,"source":"deterministic_fallback"}

def apply_aggregator_policy(aggregator,verification,consensus,feedback_used,candidate_count,context_count):
    parsed=aggregator.get("parsed_output") if aggregator.get("parse_success") else None; answer=(parsed or {}).get("answer","")
    verifier=(verification or {}).get("parsed_output") if (verification or {}).get("parse_success") else None; status=(verifier or {}).get("status","verifier_error")
    if status=="supported" and str((verifier or {}).get("best_answer","")).strip():
        answer=verifier["best_answer"]
        return canonical_answer(answer),"answered",None,True
    if status=="conflict":return "ABSTAIN","abstain","unresolved_conflict",True
    if status=="insufficient":return "ABSTAIN","abstain","verifier_insufficient_after_feedback" if feedback_used else "context_missing_support",True
    if answer and str(answer).strip().upper()!="ABSTAIN":return canonical_answer(answer),"answered",None,True
    if candidate_count==0:return "ABSTAIN","abstain","no_candidate_evidence",True
    if consensus.get("concrete_answer_count",0)==0:return "ABSTAIN","abstain","no_concrete_synthesis",True
    if not aggregator.get("parse_success"):return "","generation_error","generation_failure",False
    return "ABSTAIN","abstain","aggregator_uncertain",True

def run_pipeline(q,cfg):
    assert_inference_boundary("run_pipeline",q); started=time.perf_counter(); calls0=QWEN_CALL_COUNTER[q.qid]; repairs0=FORMAT_REPAIR_CALL_COUNTER[q.qid]
    t=time.perf_counter(); plan=plan_query(q); planner_seconds=time.perf_counter()-t
    bundle=retrieve_evidence(q,plan,retrieval_mode=cfg["retrieval_mode"],router_mode=cfg["router_mode"],use_graph=cfg["graph"])
    evidence=bundle["context_evidence"]; experts,synth,reason_timing=run_reasoning(q,evidence); consensus=synthesis_consensus(synth)
    t=time.perf_counter(); initial_verification=verify_answer(q,synth,evidence,consensus,"verifier"); verifier_seconds=time.perf_counter()-t
    final_verification=initial_verification; second_verification=None; rounds=1; added=[]; targeted=None; feedback_seconds=0.0
    retrieval_round_details=[compact_round(1,plan["retrieval_queries"],bundle)]
    reasoning_round_details=[{"round":1,"experts":experts,"cross_modal":synth,"verification":initial_verification}]
    verifier_parsed=initial_verification.get("parsed_output") if initial_verification.get("parse_success") else None
    request=deterministic_feedback_request(q,plan,synth,verifier_parsed)
    if cfg["feedback"] and request:
        feedback_start=time.perf_counter(); mod=request["modality"]; rq=request["query"]; assert_clean_retrieval_query(rq)
        old_context_ids={m:[str(d["parent_doc_id"]) for d in xs] for m,xs in evidence.items()}
        retry_plan={"retrieval_queries":build_query_records(q.question,[rq]),"normalized_subqueries":[rq],"needed_modalities":[mod],
                    "modality_confidence":{m:float(m==mod) for m in ["text","table","image"]},"routing_confidence":1.0}
        retry=retrieve_evidence(q,retry_plan,retrieval_mode=cfg["retrieval_mode"],router_mode="hard",use_graph=cfg["graph"])
        feedback_ids={str(d["parent_doc_id"]) for d in retry["retrieval_candidates"].get(mod,[])}
        merged_candidates=merge_evidence(bundle["retrieval_candidates"],retry["retrieval_candidates"])
        context_budgets=bundle["routing"]["context_budgets"]
        evidence,selection_diag,selection_stats=select_context(q.question,merged_candidates,context_budgets,plan,feedback_ids)
        bundle["retrieval_candidates"]=merged_candidates; bundle["context_evidence"]=evidence
        bundle["candidate_ids"]={m:[d["parent_doc_id"] for d in xs] for m,xs in merged_candidates.items()}; bundle["context_ids"]={m:[d["parent_doc_id"] for d in xs] for m,xs in evidence.items()}
        bundle["context_selection_diagnostics"]=selection_diag; bundle["context_selection_stats"]=selection_stats
        new_context_ids={m:[str(d["parent_doc_id"]) for d in xs] for m,xs in evidence.items()}; old_flat={x for xs in old_context_ids.values() for x in xs}; new_flat={x for xs in new_context_ids.values() for x in xs}; added=sorted(new_flat-old_flat); rounds=2
        targeted={"feedback_triggered":True,"feedback_modality":mod,"feedback_query":rq,"fallback_source":request["source"],
                  "old_context_ids":old_context_ids,"new_candidate_ids":retry["candidate_ids"],"new_context_ids":new_context_ids,
                  "new_unique_evidence_count":len(added),"feedback_new_evidence":bool(added)}
        retrieval_round_details.append(compact_round(2,retry_plan["retrieval_queries"],retry))
        experts,synth,retry_timing=run_reasoning(q,evidence); consensus=synthesis_consensus(synth)
        reason_timing["stage1_seconds"]+=retry_timing["stage1_seconds"]; reason_timing["stage2_seconds"]+=retry_timing["stage2_seconds"]
        t=time.perf_counter(); second_verification=verify_answer(q,synth,evidence,consensus,"verifier_second"); verifier_seconds+=time.perf_counter()-t; final_verification=second_verification
        reasoning_round_details.append({"round":2,"experts":experts,"cross_modal":synth,"verification":second_verification})
        feedback_seconds=time.perf_counter()-feedback_start
    t=time.perf_counter(); agg=aggregate_answer(q,synth,final_verification,consensus,cfg["agg_question"]); aggregator_seconds=time.perf_counter()-t
    candidate_count=sum(len(x) for x in bundle["retrieval_candidates"].values()); context_count=sum(len(x) for x in evidence.values())
    answer,status,abstention_reason,answer_ok=apply_aggregator_policy(agg,final_verification,consensus,bool(targeted),candidate_count,context_count)
    timing={"planner_seconds":planner_seconds,"retrieval_seconds":sum(x["retrieval_latency_seconds"] for x in retrieval_round_details),
            **reason_timing,"verifier_seconds":verifier_seconds,"feedback_seconds":feedback_seconds,"aggregator_seconds":aggregator_seconds,"total_seconds":time.perf_counter()-started}
    verifier_diag={"initial_verifier_status":initial_verification.get("effective_status"),"initial_best_answer":((initial_verification.get("parsed_output") or {}).get("best_answer","")),
                   "retry_modality":request.get("modality") if request else None,"retry_query":request.get("query") if request else None,"feedback_used":bool(targeted),
                   "new_evidence_found":bool(added),"second_verifier_status":second_verification.get("effective_status") if second_verification else None,
                   "second_best_answer":((second_verification.get("parsed_output") or {}).get("best_answer","")) if second_verification else None}
    final={"answer":answer,"status":status,"abstention_reason":abstention_reason,"answer_parse_success":answer_ok,"infrastructure_failure":not answer_ok}
    return {"qid":q.qid,"question":q.question,"inference_input_fields":["qid","question"],"planner":plan,
            "retrieval_candidates":compact_pool(bundle["retrieval_candidates"]),"context_evidence":compact_pool(evidence),
            "candidate_ids":bundle["candidate_ids"],"context_ids":bundle["context_ids"],"routing":bundle["routing"],
            "context_selection_diagnostics":bundle.get("context_selection_diagnostics",[]),"context_selection_stats":bundle.get("context_selection_stats",{}),
            "retrieval_latency_seconds":bundle["retrieval_latency_seconds"],"graph_provenance_count":len(bundle.get("graph_provenance",[])),
            "initial_verification":initial_verification,"verification":final_verification,"aggregator_raw":agg,"raw_final_output":agg.get("raw_output"),
            "parsed_final_answer":agg.get("parsed_output"),"final_decision":final,"retrieval_round_details":retrieval_round_details,
            "reasoning_round_details":reasoning_round_details,"synthesis_consensus":consensus,"verifier_diagnostics":verifier_diag,
            "stage1_outputs":experts,"stage2_answers":consensus.get("stage2_answers",[]),
            "initial_verifier_status":verifier_diag["initial_verifier_status"],"initial_verifier_best_answer":verifier_diag["initial_best_answer"],
            "feedback_used":bool(targeted),"retry_modality":verifier_diag["retry_modality"],"retry_query":verifier_diag["retry_query"],
            "feedback_candidate_ids":(targeted or {}).get("new_candidate_ids",{}),"feedback_new_evidence_count":(targeted or {}).get("new_unique_evidence_count",0),
            "feedback_context_ids":(targeted or {}).get("new_context_ids",{}),"second_verifier_status":verifier_diag["second_verifier_status"],
            "second_verifier_best_answer":verifier_diag["second_best_answer"],"aggregator_output":agg,
            "targeted_feedback":targeted,"retrieval_rounds":rounds,"new_docs_added":added,"final_answer":answer,"abstain":answer=="ABSTAIN",
            "abstention_reason":abstention_reason,"forced_parser_abstain":False,"timing":timing,"total_seconds":timing["total_seconds"],
            "qwen_call_count":QWEN_CALL_COUNTER[q.qid]-calls0,"format_repair_call_count":FORMAT_REPAIR_CALL_COUNTER[q.qid]-repairs0,"latency_seconds":timing["total_seconds"]}

In [ ]:
# Cell 23 - FULL configuration only; baselines and the oracle are intentionally not executed
CONFIGS={
 "FULL":{"planner":True,"router":True,"router_mode":"soft","retrieval_mode":"hybrid",
         "graph":True,"verifier":True,"feedback":True,"agg_question":True}
}
assert list(CONFIGS)==["FULL"]
print("Execution methods:",list(CONFIGS),"(B1, B2, and MAMMQA_ORACLE are not scheduled)")

In [ ]:
# Cell 24 - gold parsing and candidate/context retrieval metrics
def answer_strings(row):
    vals=[]
    def walk(x):
        if isinstance(x,str):vals.append(x)
        elif isinstance(x,list):
            for y in x:walk(y)
        elif isinstance(x,dict):
            for k in ["answer","answers","aliases","normalized_answer"]:
                if k in x:walk(x[k])
    walk(row.get("answers",[])); return list(dict.fromkeys(v.strip() for v in vals if v.strip()))

def support_info(row):
    ids=[]; mods={}; known={m:{str(x["id"]) for x in raw} for m,raw in [("text",texts_raw),("table",tables_raw),("image",images_raw)]}
    def add(v,mod=None):
        s=str(v); inferred=mod or next((m for m,z in known.items() if s in z),None)
        if s not in {"","None"} and inferred and s in known[inferred]:ids.append(s); mods[s]=inferred
    def walk(x,mod=None):
        if isinstance(x,list):
            for y in x:walk(y,mod)
        elif isinstance(x,dict):
            local={"text":"text","table":"table","image":"image"}.get(str(x.get("modality",x.get("doc_part",mod))).lower(),mod)
            for k,v in x.items():
                km={"text_doc_id":"text","table_id":"table","image_doc_id":"image","doc_id":local}.get(k)
                if km:add(v,km)
                elif k not in {"answer","text","content"}:walk(v,local)
        elif mod:add(x,mod)
    walk(row.get("supporting_context",[])); return list(dict.fromkeys(ids)),mods

def evaluation_only(row):
    ids,mods=support_info(row); meta=row.get("metadata",{})
    return EvaluationOnly(answer_strings(row),ids,mods,str(meta.get("type","unknown")),list(meta.get("modalities",[])),row.get("intermediate_answers"))

ARTICLES={"a","an","the"}
def norm_answer(s):
    s=str(s).lower(); s="".join(" " if c in string.punctuation else c for c in s); return " ".join(w for w in s.split() if w not in ARTICLES)
def em_f1(pred,golds):
    p=norm_answer(pred).split(); best_em=0; best_f1=0.0
    for g0 in golds:
        g=norm_answer(g0).split(); best_em=max(best_em,int(p==g)); common=Counter(p)&Counter(g); n=sum(common.values())
        score=0 if not p or not g or not n else 2*(n/len(p))*(n/len(g))/((n/len(p))+(n/len(g)))
        best_f1=max(best_f1,score)
    return best_em,best_f1

def ranked_parent_ids(result,k,mod=None,pool="retrieval_candidates"):
    pools=result.get(pool,{}) or {}; docs=pools.get(mod,[]) if mod else [x for xs in pools.values() for x in xs]
    docs=sorted(docs,key=lambda x:x.get("final_score",0),reverse=True)
    return list(dict.fromkeys(str(x["parent_doc_id"]) for x in docs))[:k]

def retrieval_scores(result,ev):
    out={}; gold=set(map(str,ev.gold_support_ids))
    for k in [1,3,5,10]:
        ids=ranked_parent_ids(result,k); hit=len(gold&set(ids)); out[f"recall@{k}"]=hit/len(gold) if gold else math.nan
        out[f"precision@{k}"]=hit/len(ids) if ids else 0.0; out[f"hit@{k}"]=int(hit>0) if gold else math.nan
        out[f"all_support_recall@{k}"]=int(gold<=set(ids)) if gold else math.nan
        for mod in ["text","table","image"]:
            gm={x for x in gold if ev.gold_support_modalities.get(x)==mod}; mids=set(ranked_parent_ids(result,k,mod))
            out[f"{mod}_recall@{k}"]=len(gm&mids)/len(gm) if gm else math.nan
    ranked=ranked_parent_ids(result,10**9); out["mrr"]=next((1/(i+1) for i,x in enumerate(ranked) if x in gold),0.0) if gold else math.nan
    candidate=set(ranked); context=set(ranked_parent_ids(result,10**9,pool="context_evidence"))
    out["gold_candidate_pool_coverage"]=len(gold&candidate)/len(gold) if gold else math.nan
    out["gold_context_coverage"]=len(gold&context)/len(gold) if gold else math.nan
    out["gold_in_candidate_pool"]=int(gold<=candidate) if gold else math.nan; out["gold_in_context"]=int(gold<=context) if gold else math.nan
    out["retrieved_document_count"]=len(candidate); out["context_document_count"]=len(context)
    gold_mods=set(ev.gold_support_modalities.values()); selected=set(result.get("planner",{}).get("needed_modalities",[]))
    out["planner_modality_recall"]=len(gold_mods&selected)/len(gold_mods) if gold_mods else math.nan
    return out

In [ ]:
# Cell 25 - reuse or deterministically freeze the disjoint VALIDATION16_V2 set before inference
DEBUG16_V1=[
 "f7dd1ac143507bd2ab9a480f18b86706","254afe636a34c3a5d94efa22d95465ad",
 "ebb69554283456251305b4b8ab579f76","4b9a4868455f98a69a8494ef1dfe7205",
 "186146ea1f5a9f7ada7f7fd31c5c75a4","c9fcae0ac6e3aa3f08375db9e4fe9037",
 "5dfd4afc175cdebf43295316a540d70c","6d372c501004c3e448b1991f39160e99",
 "b5d7685e595423638d26dd60e99fcc39","3126664c6ce707c6aca2c475ace8db08",
 "183a1eec0e38eaee513a5ec7996f55f1","2a5fdab4235e28037389777f473c4f9f",
 "2f154508615dc2d34b4d4b7fdced0b0e","22e22f3b558eeae2178ddd9cb6bc2b04",
 "1e3f1a08d865b3676a5ca81d8adc270b","ceb2b6a63d5f3b5eb8075979a55e7471"]
row_by_qid={query_only(r).qid:r for r in dev_raw}; assert all(qid in row_by_qid for qid in DEBUG16_V1)

def validation_stratum(row):
    meta=row.get("metadata",{}) or {}; qtype=str(meta.get("type","unknown")); modalities=meta.get("modalities",[]) or []
    return qtype+"|"+"+".join(sorted(map(str,modalities)))
def deterministic_validation16(rows):
    eligible=[r for r in rows if query_only(r).qid not in set(DEBUG16_V1)]; strata=defaultdict(list)
    for r in eligible:strata[validation_stratum(r)].append(r)
    rng=random.Random(VALIDATION_SELECTION_SEED)
    for values in strata.values():rng.shuffle(values)
    selected=[]
    while len(selected)<16:
        progressed=False
        for key in sorted(strata):
            if strata[key] and len(selected)<16:selected.append(strata[key].pop()); progressed=True
        if not progressed:break
    assert len(selected)==16
    return [query_only(r).qid for r in selected]

# Reuse any previously frozen set that matches the fixed debug set and seed. Conflicting
# valid frozen sets are a hard stop rather than an invitation to choose after seeing results.
frozen_candidates=[OUTPUT_ROOT/"frozen_question_sets.json",
                   BASE_OUTPUT_ROOT/"qa_targeted_repair_v2_gate_seed42"/"frozen_question_sets.json",
                   BASE_OUTPUT_ROOT/"qa_targeted_repair_v2_validation16_seed42"/"frozen_question_sets.json"]
if input_root.exists():frozen_candidates.extend(input_root.rglob("frozen_question_sets.json"))
valid_frozen=[]
for path in dict.fromkeys(Path(p) for p in frozen_candidates):
    if not path.exists():continue
    try:
        payload=json.loads(path.read_text(encoding="utf-8")); candidate=list(map(str,payload.get("VALIDATION16_V2",[])))
        seed=int(payload.get("validation_selection_seed",VALIDATION_SELECTION_SEED)); debug=list(map(str,payload.get("DEBUG16_V1",[])))
        if len(candidate)==16 and len(set(candidate))==16 and seed==VALIDATION_SELECTION_SEED and debug==DEBUG16_V1 and all(q in row_by_qid for q in candidate):
            valid_frozen.append((path,candidate))
    except Exception as exc:print("Ignoring invalid frozen-set file",path,exc)
if valid_frozen:
    unique_sets={tuple(qids) for _,qids in valid_frozen}
    assert len(unique_sets)==1,f"STOP: conflicting pre-existing VALIDATION16_V2 sets: {[str(p) for p,_ in valid_frozen]}"
    VALIDATION16_V2=list(next(iter(unique_sets))); VALIDATION_SET_SOURCE=str(valid_frozen[0][0])
else:
    VALIDATION16_V2=deterministic_validation16(dev_raw); VALIDATION_SET_SOURCE="deterministic_generation"

intersection=set(DEBUG16_V1)&set(VALIDATION16_V2)
print("DEBUG16 count =",len(DEBUG16_V1)); print("VALIDATION16_V2 count =",len(VALIDATION16_V2)); print("intersection count =",len(intersection))
assert len(DEBUG16_V1)==len(set(DEBUG16_V1))==16
assert len(VALIDATION16_V2)==len(set(VALIDATION16_V2))==16
assert not intersection,"STOP: DEBUG16_V1 and VALIDATION16_V2 overlap"
assert EXPERIMENT_PHASE=="VALIDATION16_V2" and FAST_VALIDATION is False

frozen_payload={"experiment_phase":EXPERIMENT_PHASE,"DEBUG16_V1":DEBUG16_V1,"VALIDATION16_V2":VALIDATION16_V2,
                "validation_selection_seed":VALIDATION_SELECTION_SEED,"source":VALIDATION_SET_SOURCE}
(OUTPUT_ROOT/"frozen_question_sets.json").write_text(json.dumps(frozen_payload,indent=2),encoding="utf-8")
active_qids=VALIDATION16_V2
SELECTED_BASE=[row_by_qid[qid] for qid in active_qids]; SELECTED=SELECTED_BASE[SHARD_ID::NUM_SHARDS]; SELECTED_QIDS=[query_only(r).qid for r in SELECTED]
assert NUM_SHARDS==1 and SHARD_ID==0 and SELECTED_QIDS==VALIDATION16_V2 and len(SELECTED_QIDS)==16
selected_public=[{"experiment_phase":EXPERIMENT_PHASE,"selection_seed":VALIDATION_SELECTION_SEED,
                  "qid":query_only(r).qid,"question":query_only(r).question} for r in SELECTED]
assert all(set(x)=={"experiment_phase","selection_seed","qid","question"} for x in selected_public)
(OUTPUT_ROOT/"selected_questions.json").write_text(json.dumps(selected_public,indent=2,ensure_ascii=False),encoding="utf-8")
assert all(set(asdict(query_only(r)))=={"qid","question"} for r in SELECTED)
print("VALIDATION16_V2 source =",VALIDATION_SET_SOURCE); print("selected qids =",SELECTED_QIDS)
print("LEAKAGE CHECK PASSED")

In [ ]:
# Cell 26 - gold-free synthetic unit tests and targeted change table
def synthetic_verifier_rule(agent_answers,supported_answers):
    concrete=[normalize_consensus_answer(x) for x in agent_answers if normalize_consensus_answer(x) not in {"","insufficient","none","abstain"}]
    supported={normalize_consensus_answer(x) for x in supported_answers}
    supported_concrete=sorted(set(concrete)&supported)
    if len(supported_concrete)>=2:return "conflict"
    if len(supported_concrete)==1:return "supported"
    return "insufficient"

VERIFIER_UNIT_TESTS={
 "unanimous_supported":synthetic_verifier_rule(["Andrew Lloyd Webber"]*3,["Andrew Lloyd Webber"])=="supported",
 "single_supported_neutral_insufficient":synthetic_verifier_rule(["Eddy Merckx","INSUFFICIENT","INSUFFICIENT"],["Eddy Merckx"])=="supported",
 "supported_conflict":synthetic_verifier_rule(["X","Y","INSUFFICIENT"],["X","Y"])=="conflict",
 "all_insufficient":synthetic_verifier_rule(["INSUFFICIENT"]*3,[])=="insufficient"}

synthetic_q=QueryInput("synthetic_feedback","Identify the requested visual attribute")
synthetic_plan={"normalized_subqueries":["identify logo color"],"retrieval_queries":build_query_records(synthetic_q.question,["identify logo color"]),
                "needed_modalities":["image"],"modality_confidence":{"text":.1,"table":.1,"image":.8}}
synthetic_synth={"image":{"parse_success":True,"parsed_output":{"candidate_answer":"INSUFFICIENT","supporting_doc_ids":[]},"raw_output":"<answer>INSUFFICIENT</answer>"}}
explicit=deterministic_feedback_request(synthetic_q,synthetic_plan,synthetic_synth,{"status":"insufficient","retry_modality":"image","retry_query":"identify logo color"})
fallback=deterministic_feedback_request(synthetic_q,synthetic_plan,synthetic_synth,{"status":"insufficient","retry_modality":"none","retry_query":""})
mock_calls=Counter()
def mock_image_retrieval():
    mock_calls["image"]+=1
    return [{"parent_doc_id":"new_image","doc_id":"new_image","modality":"image","final_score":1.0,"retrieval_text":"logo color blue",
             "query_provenance":[{"query_text":"identify logo color","query_source":"decomposed_subquery","rank":1,"score":1.0,"method":"hybrid"}]}]
old={"image":[{"parent_doc_id":"old_image","doc_id":"old_image","modality":"image","final_score":.5,"retrieval_text":"unrelated logo",
               "query_provenance":[{"query_text":synthetic_q.question,"query_source":"original_question","rank":1,"score":.5,"method":"hybrid"}]}]}
new_docs=mock_image_retrieval(); merged=merge_evidence(old,{"image":new_docs}); chosen,_,_=select_modality_context("image",merged["image"],1,2,synthetic_plan,{"new_image"})
FEEDBACK_UNIT_TESTS={"explicit_request":explicit=={"modality":"image","query":"identify logo color","source":"verifier"},
 "missing_fields_fallback":fallback and fallback["modality"]=="image" and bool(fallback["query"]),"retrieval_called_once":mock_calls["image"]==1,
 "new_candidates_merged":{d["parent_doc_id"] for d in merged["image"]}=={"old_image","new_image"},"context_reranked":any(d["parent_doc_id"]=="new_image" for d in chosen)}

def mock_envelope(parsed):return {"parse_success":parsed is not None,"parsed_output":parsed,"raw_output":""}
consensus_one={"concrete_answer_count":1}
a1=apply_aggregator_policy(mock_envelope({"answer":"ABSTAIN","status":"abstain"}),mock_envelope({"status":"supported","best_answer":"X"}),consensus_one,False,3,2)
a2=apply_aggregator_policy(mock_envelope({"answer":"ABSTAIN","status":"abstain"}),mock_envelope({"status":"insufficient","best_answer":""}),{"concrete_answer_count":0},True,3,2)
a3=apply_aggregator_policy(mock_envelope({"answer":"X","status":"answered"}),mock_envelope({"status":"conflict","best_answer":""}),consensus_one,True,3,2)
AGGREGATOR_UNIT_TESTS={"preserve_verifier_supported":a1[0]=="X" and a1[1]=="answered","insufficient_after_feedback_abstains":a2[0]=="ABSTAIN","conflict_after_feedback_abstains":a3[0]=="ABSTAIN"}
UNIT_TEST_RESULTS={"verifier":VERIFIER_UNIT_TESTS,"feedback":FEEDBACK_UNIT_TESTS,"aggregator":AGGREGATOR_UNIT_TESTS}
assert all(all(group.values()) for group in UNIT_TEST_RESULTS.values()),UNIT_TEST_RESULTS

assert RETRIEVAL_CANDIDATE_K==20 and MAX_INPUT_TOKENS==4096 and REPAIR_STAGES=={"aggregator"}
assert parse_compact_verifier("STATUS: supported\nBEST_ANSWER: Example\nSUPPORT_IDS: D1")
CHANGE_ROWS=[
 ["Context selector","top-k slice","query/modality-aware deterministic MMR"],["Subquery coverage","none","reserved coverage bonus"],
 ["Redundancy handling","none","token-Jaccard penalty"],["Adaptive K","fixed 3/2/2","bounded 4/3/2"],
 ["Verifier semantics","conservative agreement","one supported candidate can pass; INSUFFICIENT is neutral"],
 ["Verifier fallback","missing fields skipped feedback","deterministic modality/query fallback"],["Feedback trigger","often skipped","always once for valid insufficient/conflict"],
 ["Feedback retrieval","existing narrow context","full target-modality K=20"],["Second verification","skipped","one cheap final verifier"],
 ["Aggregator abstention","verifier insufficiency dominated","evidence-aware explicit reason"],["Consensus diagnostic","none","normalized groups/count/evidence IDs"],
 ["Gold context retention","coverage only","post-freeze lost IDs + retention"],["Cache reuse","direct paths only","recursive Kaggle cache discovery + validation"]]
CHANGE_DF=pd.DataFrame(CHANGE_ROWS,columns=["Component","Old behavior","New behavior"])
print("\nTARGETED SCIENTIFIC REPAIR\n",CHANGE_DF.to_string(index=False)); print("\nUNIT TEST RESULTS\n",json.dumps(UNIT_TEST_RESULTS,indent=2))
(DIRS["logs"]/"targeted_change_summary.json").write_text(json.dumps({"changes":[dict(zip(["component","old","new"],x)) for x in CHANGE_ROWS],"unit_tests":UNIT_TEST_RESULTS},indent=2),encoding="utf-8")
print("PRE-RUN TESTS PASSED. Engineering gate remains limited to",len(SELECTED),"questions.")

In [ ]:
# Cell 27 - frozen FULL-only VALIDATION16_V2 inference with per-question checkpointing
def append_jsonl(path,row):append_jsonl_log(path,row)
def read_jsonl_checked(path):
    path=Path(path)
    if not path.exists():return []
    rows=[]
    with path.open(encoding="utf-8") as f:
        for no,line in enumerate(f,1):
            if line.strip():
                try:rows.append(json.loads(line))
                except json.JSONDecodeError as e:raise ValueError(f"{path}:{no}: {e}") from e
    return rows
def clean_for_json(x):
    if isinstance(x,Path):return str(x)
    if isinstance(x,dict):return {k:clean_for_json(v) for k,v in x.items() if k!="source_record"}
    if isinstance(x,(list,tuple)):return [clean_for_json(v) for v in x]
    if isinstance(x,np.generic):return x.item()
    return x
def completed(path):
    rows=read_jsonl_checked(path); qids=[r["qid"] for r in rows]
    assert len(qids)==len(set(qids)); assert qids==SELECTED_QIDS[:len(qids)],f"Stale or out-of-order checkpoint: {qids}"
    return set(qids)

CONFIGS={"FULL":{"planner":True,"router":True,"router_mode":"soft","retrieval_mode":"hybrid","graph":True,"verifier":True,"feedback":True,"agg_question":True}}
METHODS=["FULL"]; prediction_name="FULL.jsonl"
FINGERPRINT_PAYLOAD={"model":MODEL_NAME,"seed":SEED,"prompts":[PLANNER_PROMPT,EXPERT_PROMPT,SYNTH_PROMPT,VERIFY_PROMPT,AGG_PROMPT],
 "retrieval":{"mode":RETRIEVAL_MODE,"router":ROUTER_MODE,"candidate_k":RETRIEVAL_CANDIDATE_K,"rrf_k":RRF_K},
 "context":{"base":[TEXT_CONTEXT_K,TABLE_CONTEXT_K,IMAGE_CONTEXT_K],"max":[TEXT_CONTEXT_K_MAX,TABLE_CONTEXT_K_MAX,IMAGE_CONTEXT_K_MAX],
            "fallback":[FALLBACK_TEXT_K,FALLBACK_TABLE_K,FALLBACK_IMAGE_K],"weights":CONTEXT_SELECTION_WEIGHTS,"duplicate_threshold":CONTEXT_DUPLICATE_THRESHOLD},
 "graph":{"enabled":ENABLE_GRAPH,"seed_k":GRAPH_SEED_K,"max_expanded":GRAPH_MAX_EXPANDED_DOCS,"max_degree":GRAPH_MAX_DEGREE},
 "generation":{"do_sample":MAMMQA_DO_SAMPLE,"temperature":MAMMQA_TEMPERATURE,"top_p":MAMMQA_TOP_P,"stage_tokens":STAGE_MAX_NEW_TOKENS,"max_input":MAX_INPUT_TOKENS},
 "verifier":{"do_sample":VERIFIER_DO_SAMPLE,"protocol":VERIFY_PROMPT},"feedback":{"candidate_k":FEEDBACK_CANDIDATE_K,"max_rounds":MAX_RETRIEVAL_ROUNDS},
 "aggregator":{"protocol":AGG_PROMPT,"repair_stages":sorted(REPAIR_STAGES)}}
IMPLEMENTATION_FINGERPRINT=hashlib.sha256(json.dumps(FINGERPRINT_PAYLOAD,sort_keys=True).encode()).hexdigest()
EXPERIMENT_CONFIG={"experiment_id":EXPERIMENT_ID,"experiment_phase":EXPERIMENT_PHASE,"implementation_fingerprint":IMPLEMENTATION_FINGERPRINT,
 "fingerprint_payload":FINGERPRINT_PAYLOAD,"validation_selection_seed":VALIDATION_SELECTION_SEED,"debug16_v1":DEBUG16_V1,
 "validation16_v2":VALIDATION16_V2,"qids":SELECTED_QIDS,"methods":METHODS,"cache_read_roots":[str(x) for x in CACHE_READ_ROOTS]}
if RUNTIME_CONFIG_PATH.exists():
    prior_config=json.loads(RUNTIME_CONFIG_PATH.read_text(encoding="utf-8"))
    assert prior_config==EXPERIMENT_CONFIG,"STOP: resumed run has a mismatching implementation/config fingerprint or qid set"
RUNTIME_CONFIG_PATH.write_text(json.dumps(EXPERIMENT_CONFIG,indent=2),encoding="utf-8")
print("IMPLEMENTATION_FINGERPRINT =",IMPLEMENTATION_FINGERPRINT)

STARTUP_SECONDS=time.perf_counter()-NOTEBOOK_START_TIME
def run_full(rows):
    path=DIRS["predictions"]/prediction_name; existing=read_jsonl_checked(path); done=completed(path); completed_times=[float(r.get("latency_seconds",0)) for r in existing]
    timed={r.get("qid") for r in read_jsonl_checked(TIMING_LOG_PATH)}
    for row in existing:
        if row["qid"] not in timed:append_jsonl(TIMING_LOG_PATH,{"qid":row["qid"],**row.get("timing",{}),"qwen_call_count":row.get("qwen_call_count"),"format_repair_call_count":row.get("format_repair_call_count")})
    print(f"[FULL] {len(done)}/{len(rows)} already complete | startup_seconds={STARTUP_SECONDS:.1f}",flush=True)
    for position,raw in enumerate(rows,1):
        q=query_only(raw)
        if q.qid in done:print(f"[FULL] {position}/{len(rows)} RESUME-SKIP {q.qid}",flush=True); continue
        started=time.perf_counter(); result=clean_for_json(run_pipeline(q,CONFIGS["FULL"])); append_jsonl(path,result); done.add(q.qid)
        elapsed=time.perf_counter()-started; completed_times.append(elapsed)
        append_jsonl(TIMING_LOG_PATH,{"qid":q.qid,**result["timing"],"qwen_call_count":result["qwen_call_count"],"format_repair_call_count":result["format_repair_call_count"]})
        complete=len(done); avg=float(np.mean(completed_times)); remaining=max(0,len(rows)-complete)*avg/60
        print(f"[FULL] {complete}/{len(rows)} complete | time_this_question={elapsed:.1f}s | average_question_time={avg:.1f}s | estimated_remaining_minutes={remaining:.1f} | qwen_calls={result['qwen_call_count']} | repair_calls={result['format_repair_call_count']}",flush=True)
        if complete%4==0:gc.collect()
    return path

assert EXPERIMENT_PHASE=="VALIDATION16_V2" and FAST_VALIDATION is False and SELECTED_QIDS==VALIDATION16_V2
assert model is not None and processor is not None and METHODS==["FULL"]
INFERENCE_STARTED=time.perf_counter(); RESULT_PATHS={"FULL":run_full(SELECTED)}; INFERENCE_SECONDS=time.perf_counter()-INFERENCE_STARTED
PREDICTION_ROWS={"FULL":read_jsonl_checked(RESULT_PATHS["FULL"])}
assert [r["qid"] for r in PREDICTION_ROWS["FULL"]]==SELECTED_QIDS and len(PREDICTION_ROWS["FULL"])==16
assert all(set(r.get("inference_input_fields",[]))=={"qid","question"} for r in PREDICTION_ROWS["FULL"])
print("\nPREDICTIONS FROZEN. Gold evaluation begins only now.")
for r in PREDICTION_ROWS["FULL"]:print(r["qid"],"=>",r["final_answer"])

In [ ]:
# Cell 28 - research-quality post-freeze VALIDATION16_V2 evaluation and reports
assert EXPERIMENT_PHASE=="VALIDATION16_V2" and len(PREDICTION_ROWS["FULL"])==16
RAW_ROWS=PREDICTION_ROWS["FULL"]; PRE_EVAL_QWEN_CALLS=dict(QWEN_CALL_COUNTER); PRE_EVAL_REPAIR_CALLS=dict(FORMAT_REPAIR_CALL_COUNTER)

def mean_valid(values):
    values=[float(v) for v in values if v is not None and not (isinstance(v,float) and math.isnan(v))]
    return float(np.mean(values)) if values else math.nan
def envelope_rounds(row,key):return [x for rr in row.get("reasoning_round_details",[]) for x in rr.get(key,{}).values() if x]
def planner_envelope(row):return row.get("planner",{}).get("planner_generation") or {}
def aggregator_envelope(row):return row.get("aggregator_raw") or {}
def initial_verifier_envelope(row):return row.get("initial_verification") or {}
def second_verifier_envelope(row):
    return row.get("verification") or {} if row.get("targeted_feedback") else {}
def usable(envelope):return bool(envelope and (envelope.get("parse_success") or str(envelope.get("raw_output","")).strip()))
def modality_health(rows,kind,modality):
    env=[]
    for row in rows:
        for rr in row.get("reasoning_round_details",[]):
            group=rr.get("experts" if kind=="stage1" else "cross_modal",{})
            if group.get(modality):env.append(group[modality])
    success=sum(usable(x) if kind=="stage1" else bool(x.get("parse_success")) for x in env)
    return {"success":success,"total":len(env),"rate":success/len(env) if env else math.nan}
def parsed_health(envelopes):
    env=[x for x in envelopes if x]; success=sum(bool(x.get("parse_success")) for x in env)
    return {"success":success,"total":len(env),"rate":success/len(env) if env else math.nan}
def verifier_status(envelope):
    parsed=envelope.get("parsed_output") if envelope and envelope.get("parse_success") else None
    return (parsed or {}).get("status","verifier_error")
def status_summary(envelopes):
    counts=Counter(verifier_status(x) for x in envelopes); total=len(envelopes)
    return {k:{"count":counts.get(k,0),"percentage":counts.get(k,0)/total if total else math.nan} for k in ["supported","insufficient","conflict","verifier_error"]}
def generation_diagnostics(rows):
    diagnostics=[]
    for row in rows:
        envelopes=[planner_envelope(row),aggregator_envelope(row),initial_verifier_envelope(row)]
        if row.get("targeted_feedback"):envelopes.append(row.get("verification") or {})
        envelopes+=envelope_rounds(row,"experts")+envelope_rounds(row,"cross_modal")
        diagnostics.extend(d for env in envelopes if env for d in env.get("generation_diagnostics",[]))
    return diagnostics

# Gold annotations are first accessed below, after the frozen-prediction assertion in Cell 27.
FULL_ROWS=[]
for prediction in RAW_ROWS:
    ev=evaluation_only(row_by_qid[prediction["qid"]]); row=copy.deepcopy(prediction); em,f1=em_f1(row["final_answer"],ev.gold_answers)
    gold=set(map(str,ev.gold_support_ids)); candidates={str(x) for xs in row.get("candidate_ids",{}).values() for x in xs}; selected={str(x) for xs in row.get("context_ids",{}).values() for x in xs}
    candidate_gold=sorted(gold&candidates); selected_gold=sorted(gold&selected); lost_gold=sorted((gold&candidates)-selected)
    retention=len(selected_gold)/len(candidate_gold) if candidate_gold else math.nan
    consensus=row.get("synthesis_consensus",{}); stage2_values=[]
    for envelope in envelope_rounds(row,"cross_modal"):
        parsed=envelope.get("parsed_output") if envelope.get("parse_success") else None
        value=str((parsed or {}).get("candidate_answer","")).strip()
        if value:stage2_values.append(value)
    gold_present=bool(row.get("abstain") and any(em_f1(v,ev.gold_answers)[0]==1 for v in stage2_values))
    row.update(gold_answers=ev.gold_answers,gold_support_ids=ev.gold_support_ids,gold_support_modalities=ev.gold_support_modalities,
               exact_match=em,token_f1=f1,retrieval_metrics=retrieval_scores(row,ev),candidate_gold_ids_found=candidate_gold,
               selected_gold_ids_found=selected_gold,lost_gold_ids=lost_gold,gold_context_retention=retention,
               gold_present_in_stage2=gold_present,answered_wrong=bool(not row.get("abstain") and not em))
    FULL_ROWS.append(row)
assert dict(QWEN_CALL_COUNTER)==PRE_EVAL_QWEN_CALLS and dict(FORMAT_REPAIR_CALL_COUNTER)==PRE_EVAL_REPAIR_CALLS,"Post-gold generation is forbidden"

def classify_failure(row):
    if row["exact_match"]:return "correct"
    if not row.get("final_decision",{}).get("answer_parse_success") or row.get("final_decision",{}).get("infrastructure_failure"):return "answer_format_failure"
    gold=set(map(str,row.get("gold_support_ids",[]))); candidates={str(x) for xs in row.get("candidate_ids",{}).values() for x in xs}; selected={str(x) for xs in row.get("context_ids",{}).values() for x in xs}
    if gold and not gold<=candidates:return "retrieval_failure"
    if gold and not gold<=selected:return "context_selection_failure"
    final_status=verifier_status(row.get("verification") or {}); initial_status=verifier_status(row.get("initial_verification") or {})
    if not row.get("abstain") and final_status=="supported":return "wrong_supported_answer"
    if row.get("gold_present_in_stage2") and final_status!="supported":return "verifier_failure"
    if row.get("targeted_feedback") and initial_status in {"insufficient","conflict"} and final_status!="supported":return "feedback_failure"
    if row.get("abstain") and not gold:return "legitimate_insufficient_abstention"
    consensus=row.get("synthesis_consensus",{})
    if consensus.get("concrete_answer_count",0)==0:return "reasoning_failure"
    if row.get("abstain") and final_status in {"insufficient","conflict"}:return "legitimate_insufficient_abstention"
    if row.get("abstain"):return "aggregation_failure"
    return "reasoning_failure"
for row in FULL_ROWS:row["failure_type"]=classify_failure(row)

def metric(name):return mean_valid([r["retrieval_metrics"].get(name) for r in FULL_ROWS])
answered=[r for r in FULL_ROWS if not r.get("abstain")]; abstained=[r for r in FULL_ROWS if r.get("abstain")]
exact_correct=sum(int(r["exact_match"]) for r in FULL_ROWS); answered_correct=sum(int(r["exact_match"]) for r in answered); answered_wrong=len(answered)-answered_correct
initial_envelopes=[initial_verifier_envelope(r) for r in FULL_ROWS]; second_envelopes=[r.get("verification") or {} for r in FULL_ROWS if r.get("targeted_feedback")]
initial_summary=status_summary(initial_envelopes); second_summary=status_summary(second_envelopes)
feedback_rows=[r for r in FULL_ROWS if r.get("targeted_feedback")]; new_evidence_count=sum(bool((r.get("targeted_feedback") or {}).get("feedback_new_evidence")) for r in feedback_rows)
feedback_success=sum(verifier_status(r.get("initial_verification") or {}) in {"insufficient","conflict"} and verifier_status(r.get("verification") or {})=="supported" for r in feedback_rows)
feedback_summary={"feedback_trigger_count":len(feedback_rows),"feedback_activation_rate":len(feedback_rows)/len(FULL_ROWS),
 "new_evidence_found_count":new_evidence_count,"feedback_success_count":int(feedback_success),
 "feedback_success_definition":"initial insufficient/conflict + feedback ran + second verifier supported",
 "feedback_success_rate":feedback_success/len(feedback_rows) if feedback_rows else math.nan}
context_summary={"gold_support_total":sum(len(r["gold_support_ids"]) for r in FULL_ROWS),
 "candidate_support_found":sum(len(r["candidate_gold_ids_found"]) for r in FULL_ROWS),
 "selected_support_retained":sum(len(r["selected_gold_ids_found"]) for r in FULL_ROWS),
 "support_lost_during_context_selection":sum(len(r["lost_gold_ids"]) for r in FULL_ROWS),
 "candidate_gold_support_coverage":metric("gold_candidate_pool_coverage"),"selected_context_gold_support_coverage":metric("gold_context_coverage"),
 "mean_gold_context_retention":mean_valid([r["gold_context_retention"] for r in FULL_ROWS])}
summary_row={"experiment":EXPERIMENT_ID,"n":len(FULL_ROWS),"exact_correct":exact_correct,"EM":exact_correct/len(FULL_ROWS),
 "mean_token_F1":mean_valid([r["token_f1"] for r in FULL_ROWS]),"answered_count":len(answered),"answer_rate":len(answered)/len(FULL_ROWS),
 "abstention_count":len(abstained),"abstention_rate":len(abstained)/len(FULL_ROWS),"answered_correct":answered_correct,
 "answered_wrong":answered_wrong,"answered_accuracy":answered_correct/len(answered) if answered else math.nan}
retrieval_row={"experiment":EXPERIMENT_ID,"n":len(FULL_ROWS),"R@1":metric("recall@1"),"R@3":metric("recall@3"),"R@5":metric("recall@5"),
 "R@10":metric("recall@10"),"MRR":metric("mrr"),**{k:context_summary[k] for k in ["candidate_gold_support_coverage","selected_context_gold_support_coverage","mean_gold_context_retention"]}}

diagnostics=generation_diagnostics(FULL_ROWS); stage2_cap_hits=sum(bool(x.get("hit_max_new_tokens")) for x in diagnostics if str(x.get("stage","")).startswith("stage2_"))
oom_count=sum(bool(x.get("oom") or x.get("oom_retried")) for x in diagnostics); protected_truncations=sum(bool(x.get("protected_prompt_truncated")) for x in diagnostics)
health={"experiment_phase":EXPERIMENT_PHASE,"questions_completed":len(FULL_ROWS),"questions_expected":16,
 "planner":parsed_health([planner_envelope(r) for r in FULL_ROWS]),
 "stage1":{"text":modality_health(FULL_ROWS,"stage1","text"),"table":modality_health(FULL_ROWS,"stage1","table"),"image":modality_health(FULL_ROWS,"stage1","image")},
 "stage2":{"text":modality_health(FULL_ROWS,"stage2","text"),"table":modality_health(FULL_ROWS,"stage2","table"),"image":modality_health(FULL_ROWS,"stage2","image")},
 "initial_verifier":parsed_health(initial_envelopes),"second_verifier":parsed_health(second_envelopes),
 "aggregator":parsed_health([aggregator_envelope(r) for r in FULL_ROWS]),"oom_count":oom_count,"protected_prompt_truncations":protected_truncations,
 "repair_count":sum(r["format_repair_call_count"] for r in FULL_ROWS),"stage2_max_token_hit_count":stage2_cap_hits,
 "mean_qwen_calls_per_question":mean_valid([r["qwen_call_count"] for r in FULL_ROWS]),"mean_runtime_seconds_per_question":mean_valid([r["timing"]["total_seconds"] for r in FULL_ROWS]),
 "verifier_calls_per_question":[1+int(bool(r.get("targeted_feedback"))) for r in FULL_ROWS],"feedback_calls":len(feedback_rows),"cache_status":CACHE_STATUS}

per_question=[]
for r in FULL_ROWS:
    rm=r["retrieval_metrics"]; consensus=r.get("synthesis_consensus",{}); vd=r.get("verifier_diagnostics",{}); tf=r.get("targeted_feedback") or {}
    per_question.append({"qid":r["qid"],"question":r["question"],"gold":json.dumps(r["gold_answers"],ensure_ascii=False),"prediction":r["final_answer"],
     "EM":r["exact_match"],"token_F1":r["token_f1"],"abstained":r["abstain"],"answered_wrong":r["answered_wrong"],
     "candidate_gold_R1":rm.get("recall@1"),"candidate_gold_R3":rm.get("recall@3"),"candidate_gold_R5":rm.get("recall@5"),"candidate_gold_R10":rm.get("recall@10"),
     "gold_support_ids":"|".join(r["gold_support_ids"]),"candidate_gold_ids_found":"|".join(r["candidate_gold_ids_found"]),
     "selected_gold_ids_found":"|".join(r["selected_gold_ids_found"]),"lost_gold_ids":"|".join(r["lost_gold_ids"]),
     "candidate_gold_support_count":len(r["candidate_gold_ids_found"]),"selected_gold_support_count":len(r["selected_gold_ids_found"]),"gold_context_retention":r["gold_context_retention"],
     "planner_parse":bool(r.get("planner",{}).get("planner_parse_success")),"stage1_usable_rate":mean_valid([modality_health([r],"stage1",m)["rate"] for m in ["text","table","image"]]),
     "stage2_parse_rate":mean_valid([modality_health([r],"stage2",m)["rate"] for m in ["text","table","image"]]),
     "stage2_answers":json.dumps(consensus.get("stage2_answers",[]),ensure_ascii=False),"stage2_consensus_answer":consensus.get("consensus_answer"),"stage2_consensus_count":consensus.get("consensus_count"),
     "initial_verifier_status":vd.get("initial_verifier_status"),"initial_verifier_best_answer":vd.get("initial_best_answer"),"feedback_used":bool(tf),
     "retry_modality":vd.get("retry_modality"),"retry_query":vd.get("retry_query"),"feedback_candidate_ids":json.dumps(tf.get("new_candidate_ids",{})),
     "feedback_new_evidence_count":tf.get("new_unique_evidence_count",0),"feedback_context_ids":json.dumps(tf.get("new_context_ids",{})),
     "second_verifier_status":vd.get("second_verifier_status"),"second_verifier_best_answer":vd.get("second_best_answer"),
     "aggregator_parse":bool(aggregator_envelope(r).get("parse_success")),"abstention_reason":r.get("abstention_reason"),"gold_present_in_stage2":r["gold_present_in_stage2"],
     "failure_type":r["failure_type"],"total_seconds":r["timing"]["total_seconds"],"qwen_calls":r["qwen_call_count"],"repair_calls":r["format_repair_call_count"]})

failure_rows=[{"qid":r["qid"],"prediction":r["final_answer"],"abstained":r["abstain"],"answered_wrong":r["answered_wrong"],
               "abstention_reason":r.get("abstention_reason"),"failure_type":r["failure_type"],"gold_present_in_stage2":r["gold_present_in_stage2"]} for r in FULL_ROWS]
verifier_summary={"initial":initial_summary,"after_feedback":second_summary,"initial_total":len(initial_envelopes),"second_total":len(second_envelopes)}
failure_counts=dict(Counter(r["failure_type"] for r in FULL_ROWS)); abstention_reasons={k:sum(r.get("abstention_reason")==k for r in abstained) for k in ["no_candidate_evidence","context_missing_support","no_concrete_synthesis","verifier_insufficient_after_feedback","unresolved_conflict","aggregator_uncertain","generation_failure"]}

pd.DataFrame([summary_row]).to_csv(DIRS["metrics"]/"summary.csv",index=False)
pd.DataFrame([retrieval_row]).to_csv(DIRS["metrics"]/"retrieval_summary.csv",index=False)
pd.DataFrame(per_question).to_csv(DIRS["metrics"]/"per_question_FULL.csv",index=False)
pd.DataFrame(failure_rows).to_csv(DIRS["metrics"]/"failure_analysis.csv",index=False)
(DIRS["metrics"]/"pipeline_health.json").write_text(json.dumps(health,indent=2,allow_nan=True),encoding="utf-8")
(DIRS["metrics"]/"verifier_summary.json").write_text(json.dumps(verifier_summary,indent=2,allow_nan=True),encoding="utf-8")
(DIRS["metrics"]/"feedback_summary.json").write_text(json.dumps(feedback_summary,indent=2,allow_nan=True),encoding="utf-8")
(DIRS["metrics"]/"context_retention_summary.json").write_text(json.dumps(context_summary,indent=2,allow_nan=True),encoding="utf-8")

VALIDATION16_V2_COMPLETE=len(FULL_ROWS)==16 and [r["qid"] for r in FULL_ROWS]==VALIDATION16_V2
required_files=["summary.csv","retrieval_summary.csv","per_question_FULL.csv","failure_analysis.csv","pipeline_health.json","verifier_summary.json","feedback_summary.json","context_retention_summary.json"]
VALIDATION16_V2_EVALUATED=VALIDATION16_V2_COMPLETE and all((DIRS["metrics"]/x).exists() for x in required_files)
assert not (set(DEBUG16_V1)&set(VALIDATION16_V2))

def pct(x):return "N/A" if x is None or (isinstance(x,float) and math.isnan(x)) else f"{100*x:.2f}%"
print("\n============================================================\nVALIDATION16_V2 RESULTS\n============================================================")
print("Dataset:\nMMQA dev\n\nQuestions:\n16\n\nDisjoint from DEBUG16_V1:\nYES\n\nSeed:\n",VALIDATION_SELECTION_SEED)
print("\n------------------------------------------------------------\nQA\n------------------------------------------------------------")
print("Exact Match:",pct(summary_row["EM"])); print("Token F1:",pct(summary_row["mean_token_F1"])); print("Answered:",len(answered),"/ 16 ("+pct(summary_row["answer_rate"])+")")
print("Abstained:",len(abstained),"/ 16 ("+pct(summary_row["abstention_rate"])+")"); print("Answered correct:",answered_correct); print("Answered wrong:",answered_wrong); print("Answered accuracy:",pct(summary_row["answered_accuracy"]))
print("\n------------------------------------------------------------\nRETRIEVAL\n------------------------------------------------------------")
for key in ["R@1","R@3","R@5","R@10","MRR"]:print(key+":",pct(retrieval_row[key]))
print("Candidate support coverage:",pct(context_summary["candidate_gold_support_coverage"])); print("Selected-context support coverage:",pct(context_summary["selected_context_gold_support_coverage"])); print("Mean gold-context retention:",pct(context_summary["mean_gold_context_retention"]))
print("\n------------------------------------------------------------\nVERIFIER\n------------------------------------------------------------\nInitial:")
for key in ["supported","insufficient","conflict","verifier_error"]:print(key,"=",initial_summary[key]["count"],"("+pct(initial_summary[key]["percentage"])+")")
print("After feedback:")
for key in ["supported","insufficient","conflict","verifier_error"]:print(key,"=",second_summary[key]["count"],"("+pct(second_summary[key]["percentage"])+")")
print("\n------------------------------------------------------------\nFEEDBACK\n------------------------------------------------------------")
print("Triggered:",feedback_summary["feedback_trigger_count"]); print("Activation rate:",pct(feedback_summary["feedback_activation_rate"])); print("New evidence found:",feedback_summary["new_evidence_found_count"])
print("Resolved to supported:",feedback_summary["feedback_success_count"]); print("Feedback success rate:",pct(feedback_summary["feedback_success_rate"]))
print("\n------------------------------------------------------------\nABSTENTION\n------------------------------------------------------------")
print("Total:",len(abstained)); print("Reasons:")
for key,value in abstention_reasons.items():print(key,"=",value)
print("Gold answer present in Stage-II despite abstention:",sum(r["gold_present_in_stage2"] for r in abstained))
print("\nFailure taxonomy:",failure_counts)
print("\n------------------------------------------------------------\nPIPELINE HEALTH\n------------------------------------------------------------")
print("Planner:",health["planner"]); print("Stage-I usable:",health["stage1"]); print("Stage-II:",health["stage2"]); print("Verifier:",health["initial_verifier"],"second:",health["second_verifier"]); print("Aggregator:",health["aggregator"])
print("OOM:",oom_count); print("Protected truncations:",protected_truncations); print("Stage-II token-cap hits:",stage2_cap_hits); print("Average Qwen calls/question:",round(health["mean_qwen_calls_per_question"],3)); print("Average runtime/question:",round(health["mean_runtime_seconds_per_question"],3)); print("Repair calls:",health["repair_count"])
print("\nOLD DEVELOPMENT RUN reference only: metrics came from a different 16-question sample; differences are descriptive, not an apples-to-apples improvement claim.")
print("Old candidate/context coverage: 76.04% / 23.96%; new:",pct(context_summary["candidate_gold_support_coverage"]),"/",pct(context_summary["selected_context_gold_support_coverage"]))
print("\nVALIDATION16_V2_COMPLETE =",VALIDATION16_V2_COMPLETE); print("VALIDATION16_V2_EVALUATED =",VALIDATION16_V2_EVALUATED)
print("============================================================")